[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antoniovfonseca/summarize-change-components/blob/main/notebooks/v3/01_agentic_ai_quest_metrics.ipynb)

# **QUEST Engine: Agentic AI for spatial analyses**

---

## Prerequisites & Setup
To run the agentic AI workflow, prepare the following before executing the notebook cells:

1. **[Gemini API Key](https://aistudio.google.com/app/apikey)**: Store the key as a secret named `GEMINI_API_KEY`. In Colab, use the key icon in the left panel. Locally, use a `.env` file next to this notebook or an environment variable. Never paste the key in a cell.

2. **Input rasters**: One single-band GeoTIFF per time point, with file names that follow the pattern described in Section 2. In Colab, the rasters are uploaded with cell 2.3.

---

## Agent Capabilities & Available Products
Once the environment is set up, you can interact with the **QUEST Chat Interface** to request raw data, tables, charts, or spatial maps. Some of the analytical products you can extract from your time series raster maps include:
*   **Class Size**: Request the class area over time by asking for 'time series spatial extents', 'pixel counts', or 'stacked charts' of class sizes.
*   **Number of Changes**: Request the number of changes in 'time intervals' or 'overall' (in 'number of pixels' or 'percentage of the study area'),
*   **Trajectories**: Request the five types of trajectories in 'time intervals' or 'overall' (in 'number of pixels' or 'percentage of the study area').
*   **Transition Matrices**: Tabular data and heatmaps for 'time intervals', 'extent', 'allocation exchange', 'quantity allocation shift', 'alternation exchange', 'alternation shift', 'indirect extent', the five trajectories, or specific intervals (e.g., 'interval 1990-2000').
*   **Change Components**: Stacked bar charts ('time intervals', 'overall', and 'per-class') detailing the Allocation and Alternation logic decomposed into Quantity, Exchange, and Shift components.
*   **Spatial Maps**: High-resolution rendered maps for the Number of Changes and Trajectories across the study area.

## 1. Environment Setup
This section loads the spatial analysis libraries, the QUEST core engine, and the Gemini API key.

In [ ]:
# @title 1.1 Install libraries
!pip install -q matplotlib-scalebar
!pip install -q matplotlib-map-utils

In [ ]:
# @title 1.2 Import libraries
# 1. Standard Library Imports
import contextlib
import functools
import json
import os
import re
import shutil
import time
from datetime import datetime
from pathlib import Path
from typing import Dict

# 2. Third-Party Data and Geospatial Libraries
import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS, Geod, Transformer
from sklearn.metrics import confusion_matrix
import rasterio.enums

# 3. Visualization Libraries
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib_map_utils import north_arrow

# 4. Google AI
from google import genai
from google.genai import types

try:
    from google.colab import userdata
except ImportError:
    userdata = None

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None


def resolve_gemini_api_key() -> str | None:
    """Resolve the Gemini API key in the same way in Colab and VS Code."""
    if load_dotenv is not None:
        load_dotenv(Path(".env"), override=False)

    key = os.getenv("GEMINI_API_KEY")
    if key:
        return key

    if userdata is not None:
        try:
            key = userdata.get("GEMINI_API_KEY")
            if key:
                return key
        except Exception:
            pass

    return None



# 5. QUEST core engine, shared with the 03_quest_pie notebook. Locally, the
#    file sits next to this notebook. In Colab, it is downloaded from GitHub.
QUEST_CORE_URL = (
    "https://raw.githubusercontent.com/antoniovfonseca/"
    "summarize-change-components/main/notebooks/v3/quest_core.py"
)

if not os.path.exists(
    "quest_core.py",
):
    import urllib.request

    urllib.request.urlretrieve(
        QUEST_CORE_URL,
        "quest_core.py",
    )

import importlib

import quest_core

importlib.reload(
    quest_core,
)

from quest_core import (
    build_quest_tables,
    compute_all_quest_outputs,
    compute_change_components_table,
    compute_display_pixel_size_km,
    format_scale_bar_label,
    save_quest_tables,
    summarize_change_components,
)

In [ ]:
# @title 1.3 Gemini API key
# @markdown Reads the Gemini API key from a secret named GEMINI_API_KEY. In Colab, add it with the key icon in the left panel. Locally, use a .env file next to this notebook or an environment variable. Never paste the key in a cell.
api_key = resolve_gemini_api_key()

if api_key is None:
    raise RuntimeError(
        "GEMINI_API_KEY not found. "
        "In Colab, add it as a secret with the key icon in the left panel. "
        "Locally, create a .env file in the same folder as this notebook with:\n"
        "GEMINI_API_KEY=YOUR_KEY\n"
        "or run in the terminal before starting the notebook kernel:\n"
        "export GEMINI_API_KEY='YOUR_KEY'"
    )

os.environ["GEMINI_API_KEY"] = api_key

## **2. User Input**

Fill in the cells below with your data. No programming knowledge is needed. Just replace the values between the quotes.

### What you need
1. **One raster map per time point**, as a single-band GeoTIFF.
2. **A consistent file name pattern:** `{PREFIX}{YEAR}{SUFFIX}`, where `YEAR` has four digits. For example, prefix `vcr_`, year `1990`, and suffix `.tif` produce `vcr_1990.tif`. Files in the input folder that do not follow the pattern are ignored.
3. **The rasters in the input folder.** In Colab, run cell 2.3 to upload them. Locally, set `INPUT_FOLDER_PATH` to the folder that holds them and skip cell 2.3.

### Requirements for the rasters themselves
- **Identical pixel grid across every file:** same width, height, CRS, extent, and spatial resolution. If the grids do not line up pixel for pixel, the analysis will be wrong without any warning.
- **Single band.** Only band 1 of each file is read.
- **Integer class codes**, with one consistent no data value across every file marking invalid or masked pixels.

### Class definitions
Cell 2.2 defines `class_labels_dict`: the class codes found in your rasters, their display names, and the colors used in charts and maps. Add, remove, or rename entries to match your classification scheme, keeping the same syntax. A location enters the analysis only if it holds a listed class and a valid value in every year, so a code left out of `class_labels_dict` removes those locations from every product.

In [ ]:
# @title 2.1 Configure your data { display-mode: "form" }
# @markdown ### 📂 Path Settings
# @markdown Folder that holds the input rasters. In Colab, cell 2.3 uploads them here:
INPUT_FOLDER_PATH = "/content/quest_input" #@param {type:"string"}
# @markdown Folder for the rasters, tables, charts, and maps produced by QUEST:
OUTPUT_PATH = "/content/quest_output" #@param {type:"string"}

# @markdown ### 🏷️ File Names
# @markdown Example: prefix "vcr_" + year 1990 + suffix ".tif" -> "vcr_1990.tif". Leave the prefix empty for files named only by year, such as "1990.tif".
FILE_PREFIX = "" #@param {type:"string"}
FILE_SUFFIX = ".tif" #@param {type:"string"}

# @markdown ### 🗓️ Time Points
# @markdown Comma-separated years, or a Python range expression, in chronological order. Leave empty to use every year found in the input folder:
YEARS = "range(1985, 2022)" #@param {type:"string"}

# @markdown ### 🧭 Raster Settings
# @markdown Pixel value marking invalid / masked data:
NO_DATA_VALUE = 255 #@param {type:"integer"}

# @markdown ### 🤖 Agent Settings
# @markdown Gemini model used by the chat agent:
GEMINI_MODEL = "gemini-3.5-flash-lite" #@param {type:"string"}
# @markdown Show the tools called by the agent after each answer:
SHOW_TOOL_CALLS = True #@param {type:"boolean"}
# @markdown Maximum model requests per minute of your API key, 0 for no limit. The free tier allows 15 for gemini-3.5-flash-lite and 5 for gemini-3.6-flash:
REQUESTS_PER_MINUTE = 15 #@param {type:"integer"}

In [ ]:
# @title 2.2 Class Definitions
# Class code, name, and color of each class in the input rasters
class_labels_dict = {
    1: {"name": "Trees", "color": "#228B22"},
    2: {"name": "Marsh", "color": "#A0522D"},
    3: {"name": "Beach", "color": "#DEB887"},
    4: {"name": "Water", "color": "#4A90E2"},
}

In [ ]:
# @title 2.3 Upload the rasters (Colab only)
# @markdown Run this cell in Colab to upload the rasters into INPUT_FOLDER_PATH. Select every GeoTIFF of the time series at once. Skip it when the rasters are already in that folder.
from google.colab import files

os.makedirs(
    INPUT_FOLDER_PATH,
    exist_ok=True,
)

uploaded = files.upload()

# Colab saves each upload in the working folder, so move it to the input folder
for file_name in uploaded:
    destination = os.path.join(
        INPUT_FOLDER_PATH,
        file_name,
    )
    if os.path.abspath(file_name) != os.path.abspath(destination):
        shutil.move(
            file_name,
            destination,
        )

print(
    f"{len(uploaded)} files uploaded to {INPUT_FOLDER_PATH}",
)

In [ ]:
# @title 2.4 Check the input rasters
# @markdown Finds the rasters that follow the file name pattern, checks that every requested year has a file, checks that the rasters can be compared pixel by pixel, and stores the settings used by the QUEST engine.


def check_input_rasters(
    raster_by_year: dict,
    no_data_value: int,
    class_labels_dict: dict,
    sample_dim: int = 2000,
) -> dict:
    """
    Check that the input rasters can be analyzed together by QUEST.

    Problems that would make the analysis wrong raise an error. Problems
    that change what enters the analysis are reported as warnings.

    Parameters
    ----------
    raster_by_year : dict
        Mapping from year to raster path, in chronological order.
    no_data_value : int
        Pixel value marking invalid or masked data.
    class_labels_dict : dict
        Mapping from class code to metadata with "name" and "color".
    sample_dim : int, optional
        Longer side, in pixels, of the decimated sample read from each
        raster to list its values. Default is 2000.

    Returns
    -------
    dict
        Grid description, values found in the sample, and warnings.
    """
    errors = []
    warnings = []
    values_found = set()
    reference = None

    # 1. The engine stores class codes and NoData as 8-bit integers
    if not 0 <= no_data_value <= 255:
        errors.append(
            f"NO_DATA_VALUE is {no_data_value}. The QUEST engine supports "
            f"values from 0 to 255.",
        )

    codes_out_of_range = sorted(
        code
        for code in class_labels_dict
        if not 0 <= code <= 255
    )
    if codes_out_of_range:
        errors.append(
            f"Class codes {codes_out_of_range} are outside 0 to 255, the "
            f"range supported by the QUEST engine.",
        )

    for year, path in raster_by_year.items():

        # 2. Open the raster and read its grid
        try:
            src = rasterio.open(
                path,
            )
        except Exception as error:
            errors.append(
                f"{year}: the file cannot be opened as a raster ({error}).",
            )
            continue

        with src:
            grid = {
                "width": src.width,
                "height": src.height,
                "crs": src.crs,
                "transform": src.transform,
            }

            if reference is None:
                reference = dict(
                    grid,
                    year=year,
                )

            # 3. Every raster must share the grid of the first year
            for key in [
                "width",
                "height",
                "crs",
                "transform",
            ]:
                if grid[key] != reference[key]:
                    errors.append(
                        f"{year}: {key} differs from {reference['year']}, so "
                        f"the rasters do not line up pixel by pixel.",
                    )

            if not np.issubdtype(
                np.dtype(src.dtypes[0]),
                np.integer,
            ):
                errors.append(
                    f"{year}: band 1 is {src.dtypes[0]}. Class codes must be "
                    f"integers.",
                )
                continue

            if src.count > 1:
                warnings.append(
                    f"{year}: the file has {src.count} bands. Only band 1 is "
                    f"read.",
                )

            if src.nodata is not None and src.nodata != no_data_value:
                warnings.append(
                    f"{year}: the file declares NoData as {src.nodata:g}, but "
                    f"NO_DATA_VALUE is {no_data_value}. Only {no_data_value} "
                    f"is treated as NoData.",
                )

            # 4. List the values in a decimated sample of band 1
            downsample = max(
                1,
                max(src.height, src.width) // sample_dim,
            )
            sample = src.read(
                1,
                out_shape=(
                    max(1, src.height // downsample),
                    max(1, src.width // downsample),
                ),
                resampling=rasterio.enums.Resampling.nearest,
            )
            values_found.update(
                int(value)
                for value in np.unique(sample)
            )

    values_found.discard(
        no_data_value,
    )

    # 5. Values outside 0 to 255 cannot be stored by the engine
    values_out_of_range = sorted(
        value
        for value in values_found
        if not 0 <= value <= 255
    )
    if values_out_of_range:
        errors.append(
            f"Values {values_out_of_range[:10]} are outside 0 to 255, the "
            f"range supported by the QUEST engine.",
        )

    if errors:
        raise ValueError(
            "The input rasters cannot be analyzed:\n- " + "\n- ".join(errors),
        )

    # 6. Compare the values found with the legend
    unlisted_values = sorted(
        values_found - set(class_labels_dict),
    )
    if unlisted_values:
        warnings.append(
            f"Values {unlisted_values} appear in the rasters but not in "
            f"class_labels_dict. Locations holding them in any year are left "
            f"out of the analysis.",
        )

    absent_codes = sorted(
        set(class_labels_dict) - values_found,
    )
    if absent_codes:
        warnings.append(
            f"Class codes {absent_codes} are in class_labels_dict but were "
            f"not found in the sample of the rasters.",
        )

    return {
        "width": reference["width"],
        "height": reference["height"],
        "crs": str(reference["crs"]),
        "spatial_resolution": (
            abs(reference["transform"].a),
            abs(reference["transform"].e),
        ),
        "values_found_in_sample": sorted(values_found),
        "warnings": warnings,
    }


# 1. Find the files named {FILE_PREFIX}{YEAR}{FILE_SUFFIX} in the input folder
if not os.path.isdir(
    INPUT_FOLDER_PATH,
):
    raise FileNotFoundError(
        f"Input folder not found: {INPUT_FOLDER_PATH}",
    )

name_pattern = re.compile(
    rf"^{re.escape(FILE_PREFIX)}(\d{{4}}){re.escape(FILE_SUFFIX)}$",
)

rasters_found = {}
for file_name in sorted(
    os.listdir(
        INPUT_FOLDER_PATH,
    ),
):
    match = name_pattern.match(
        file_name,
    )
    if match:
        rasters_found[int(match.group(1))] = os.path.join(
            INPUT_FOLDER_PATH,
            file_name,
        )

if not rasters_found:
    raise FileNotFoundError(
        f"No file in {INPUT_FOLDER_PATH} follows the pattern "
        f"{FILE_PREFIX}YYYY{FILE_SUFFIX}.",
    )

# 2. Parse the years: a "range(a, b)" expression, a comma-separated list,
#    or empty to use every year found
years_input = str(YEARS).strip()

if not years_input:
    years = sorted(rasters_found)
elif years_input.startswith("range(") and years_input.endswith(")"):
    range_args = [
        int(part.strip())
        for part in years_input[len("range("):-1].split(",")
    ]
    years = list(range(*range_args))
else:
    years = [
        int(year.strip())
        for year in years_input.split(",")
        if year.strip()
    ]

if years != sorted(set(years)):
    raise ValueError(
        "YEARS must be unique and in chronological order.",
    )

missing_years = [
    year
    for year in years
    if year not in rasters_found
]
if missing_years:
    raise FileNotFoundError(
        f"No raster for the years {missing_years}. Expected files such as "
        f"{FILE_PREFIX}{missing_years[0]}{FILE_SUFFIX} in {INPUT_FOLDER_PATH}.",
    )

if len(years) < 2:
    raise ValueError(
        "QUEST needs at least two time points.",
    )

raster_by_year = {
    year: rasters_found[year]
    for year in years
}

# 3. Check that the rasters can be compared pixel by pixel
input_report = check_input_rasters(
    raster_by_year=raster_by_year,
    no_data_value=NO_DATA_VALUE,
    class_labels_dict=class_labels_dict,
)

# 4. Store the settings read by every QUEST module
QUEST_CONFIG = {
    "raster_by_year": raster_by_year,
    "no_data_value": NO_DATA_VALUE,
    "class_labels_dict": class_labels_dict,
    "output_path": OUTPUT_PATH,
    "input_report": input_report,
}

print(
    f"{len(years)} rasters found, from {years[0]} to {years[-1]}: "
    f"{', '.join(str(year) for year in years)}",
)
print(
    f"Grid: {input_report['width']} x {input_report['height']} pixels, "
    f"{input_report['crs']}, spatial resolution "
    f"{input_report['spatial_resolution'][0]:g} x "
    f"{input_report['spatial_resolution'][1]:g}",
)
print(
    f"Values found in a sample of the rasters: "
    f"{input_report['values_found_in_sample']}",
)
for warning in input_report["warnings"]:
    print(
        f"Warning: {warning}",
    )

## 3. Core Architecture

This section defines the base class of every module. It reads the settings from Section 2 and runs the QUEST core engine, which applies the common valid mask so that every product describes the same locations.

In [ ]:
# @title 3.1 Core Engine Initialization
# @markdown Reads the settings from Section 2 and runs the QUEST core engine once per set of years. Every module reads the same cached results, so all products use the same common valid mask: a location enters the analysis only if it is valid and belongs to a listed class in every year.

# Results of the core engine, shared by every module and keyed by the
# ordered tuple of raster paths, so each set of years is computed once
_QUEST_RESULTS_CACHE = {}


class QuestBase:
    """
    Core engine for spatial analysis of a time series of raster maps.

    The computation is done by `quest_core.compute_all_quest_outputs`, the
    same engine used by the 03_quest_pie notebook. It runs the first time a
    module needs a result and is cached for every later question.

    Parameters
    ----------
    config : dict
        Settings built by cell 2.4, with the keys "raster_by_year",
        "no_data_value", "class_labels_dict" and "output_path".

    Examples
    --------
    >>> base = QuestBase(config=QUEST_CONFIG)
    >>> results = base.get_quest_results()
    """

    def __init__(
        self,
        config: dict,
    ):
        self.config = config
        self.no_data_value = config["no_data_value"]
        self.output_path = config["output_path"]
        self.class_labels_dict = config["class_labels_dict"]
        self.class_labels = [
            self.class_labels_dict[key]["name"]
            for key in sorted(self.class_labels_dict.keys())
        ]

        # Keep the rasters in chronological order, so the order of the
        # paths always matches the order of the years
        self.raster_by_year = dict(
            config["raster_by_year"],
        )
        self.image_paths = [
            self.raster_by_year[year]
            for year in sorted(self.raster_by_year)
        ]

    def restrict_years(
        self,
        start_year: int,
        end_year: int,
    ) -> None:
        """
        Keep only the rasters from `start_year` to `end_year`, inclusive.

        The common valid mask and every product are then computed for that
        time extent only.

        Parameters
        ----------
        start_year : int
            First year of the time extent.
        end_year : int
            Last year of the time extent.
        """
        self.raster_by_year = {
            year: path
            for year, path in self.raster_by_year.items()
            if start_year <= year <= end_year
        }
        self.image_paths = [
            self.raster_by_year[year]
            for year in sorted(self.raster_by_year)
        ]

    @property
    def years(self) -> list[int]:
        """Years of the rasters in use, in chronological order."""
        return sorted(
            self.raster_by_year,
        )

    def get_quest_results(self) -> dict:
        """
        Run the QUEST core engine once for the rasters in use and cache the
        results for every module.

        Returns
        -------
        dict
            "results" : output of `quest_core.compute_all_quest_outputs`.
            "tables" : output of `quest_core.build_quest_tables`.
            "components" : output of
                `quest_core.compute_change_components_table`.
            "study_area_pixels" : int, number of pixels in the common valid
                mask.
            "run_dir" : str, folder of the outputs for these years.
        """
        if len(self.image_paths) < 2:
            raise ValueError(
                "At least two time points are needed. "
                f"Years in use: {self.years}.",
            )

        cache_key = (
            tuple(self.image_paths),
            tuple(sorted(self.class_labels_dict)),
        )

        if cache_key not in _QUEST_RESULTS_CACHE:
            years = self.years
            run_dir = os.path.join(
                self.output_path,
                f"{years[0]}-{years[-1]}",
            )
            results = compute_all_quest_outputs(
                image_paths=self.image_paths,
                output_dir=run_dir,
                years=years,
                class_dict=self.class_labels_dict,
                nodata_val=self.no_data_value,
            )
            tables = build_quest_tables(
                quest_results=results,
                years=years,
            )
            save_quest_tables(
                tables=tables,
                output_dir=run_dir,
            )
            components = compute_change_components_table(
                tables=tables,
                class_labels_dict=self.class_labels_dict,
            )
            components.to_csv(
                os.path.join(
                    run_dir,
                    "tables",
                    "change_components.csv",
                ),
                index=False,
            )
            _QUEST_RESULTS_CACHE[cache_key] = {
                "results": results,
                "tables": tables,
                "components": components,
                "study_area_pixels": int(
                    results["class_counts"][0].sum(),
                ),
                "run_dir": run_dir,
            }

        return _QUEST_RESULTS_CACHE[cache_key]

## 4. Modules

This section contains the modules of the QUEST engine. The available modules are Class Size, Number of Changes, Trajectory, Transition Matrices, Change Components, and Maps.

### 4.1 Class Size

In [ ]:
# @title 4.1.1 Class Size Processing Module
# @markdown Houses the computational methods to tally valid pixel distributions and the Matplotlib logic required by the AI agent to render on-demand charts.

class QuestClassSize(QuestBase):
    """
    Isolates the first core product of the QUEST methodology. This class inherits
    cloud connections and the strict temporal mask from the base engine, opening
    each annual raster map to extract the physical footprint of every land cover
    category. It counts only valid pixels across decades to build a precise
    historical timeline of environmental expansion or shrinkage, while integrating
    a built-in visualization engine that translates numerical dictionaries into
    dynamically scaled, sorted stacked bar charts.

    Parameters
    ----------
    config : dict
        Settings built by cell 2.4.

    Examples
    --------
    >>> class_size_engine = QuestClassSize(config=QUEST_CONFIG)
    >>> pixel_data = class_size_engine.compute_class_pixels_over_time()
    """
    def compute_class_pixels_over_time(self) -> dict:
        """
        Returns the pixel count for each land cover class at each year,
        inside the common valid mask, so every year has the same total.

        Returns
        -------
        dict
            A nested dictionary mapping years to class pixel counts.

        Examples
        --------
        >>> pixel_data = class_size_engine.compute_class_pixels_over_time()
        """
        cached = self.get_quest_results()
        results = cached["results"]
        class_ids = results["class_ids"]

        output = {}
        for year, year_counts in zip(
            self.years,
            results["class_counts"],
        ):
            counts_by_id = dict(
                zip(
                    class_ids,
                    year_counts,
                ),
            )
            output[year] = {
                info["name"]: int(
                    counts_by_id.get(
                        class_id,
                        0,
                    ),
                )
                for class_id, info in self.class_labels_dict.items()
            }

        return output

    def plot_class_sizes(
        self,
        specific_years: list[int] = None,
    ) -> str:
        """
        Generates and displays a stacked bar chart of pixel counts per class,
        sorting categories based on net area change and applying adaptive scaling.

        Parameters
        ----------
        specific_years : list of int, optional
            A list of specific years to include in the plot. If None, uses all available years.

        Returns
        -------
        str
            Confirmation string indicating the chart was rendered on screen.

        Examples
        --------
        >>> class_size_engine.plot_class_sizes(specific_years=[1985, 2000, 2021])
        """
        data_dict = self.compute_class_pixels_over_time()

        # 1. Filter dictionary if the user requested specific years
        if specific_years:
            data_dict = {
                year: counts
                for year, counts in data_dict.items()
                if year in specific_years
            }

        df = pd.DataFrame.from_dict(
            data_dict,
            orient="index",
        )
        df.index.name = "Year"
        years_array = df.index.values

        # 2. Determine Y-axis scaling factor and label
        max_val = df.to_numpy().max()

        if max_val >= 1_000_000_000_000:
            scale_factor = 1_000_000_000_000
            y_label = "Class Size (trillion pixels)"
        elif max_val >= 1_000_000_000:
            scale_factor = 1_000_000_000
            y_label = "Class Size (billion pixels)"
        elif max_val >= 1_000_000:
            scale_factor = 1_000_000
            y_label = "Class Size (million pixels)"
        elif max_val >= 1_000:
            scale_factor = 1_000
            y_label = "Class Size (thousand pixels)"
        elif max_val >= 100:
            scale_factor = 100
            y_label = "Class Size (hundred pixels)"
        else:
            scale_factor = 1
            y_label = "Class Size (pixels)"

        df_scaled = df / scale_factor

        # 3. Prepare color map and sorting logic
        class_ids_plot = sorted(
            self.class_labels_dict.keys(),
        )

        color_map = {
            self.class_labels_dict[c_id]["name"]: self.class_labels_dict[c_id]["color"]
            for c_id in class_ids_plot
        }

        first_year = years_array[0]
        last_year = years_array[-1]

        net_change_per_class = (
            df_scaled.loc[last_year]
            - df_scaled.loc[first_year]
        )

        name_to_id_map = {
            v["name"]: k
            for k, v in self.class_labels_dict.items()
        }

        df_sorting = net_change_per_class.to_frame(
            name="net_change",
        )
        df_sorting["class_id"] = df_sorting.index.map(
            name_to_id_map,
        )

        classes_for_stack = list(
            df_sorting.sort_values(
                by=[
                    "net_change",
                    "class_id",
                ],
                ascending=[
                    False,
                    False,
                ],
            ).index,
        )

        classes_for_legend = list(
            reversed(classes_for_stack),
        )

        # 4. Generate the Stacked Bar Chart
        fig, ax = plt.subplots(
            figsize=(
                10,
                6,
            ),
        )

        x = np.arange(
            len(years_array),
        )
        width = 0.9
        base = np.zeros(
            len(years_array),
            dtype=float,
        )
        patches_by_class = {}

        for cls in classes_for_stack:
            if cls not in df_scaled.columns:
                continue

            values_cls = df_scaled[cls].reindex(
                years_array,
                fill_value=0.0,
            ).values

            bars = ax.bar(
                x,
                values_cls,
                bottom=base,
                width=width,
                label=cls,
                color=color_map.get(
                    cls,
                    "gray",
                ),
            )
            patches_by_class[cls] = bars[0]
            base += values_cls

        # 5. Configure Axes
        ax.set_xticks(
            x,
        )
        ax.set_xticklabels(
            years_array,
        )

        n_labels = len(years_array)

        if n_labels <= 6:
            rotation = 0
            ha = "center"
        else:
            rotation = 90
            ha = "center"

        plt.setp(
            ax.get_xticklabels(),
            rotation=rotation,
            ha=ha,
        )

        ax.tick_params(
            axis="both",
            labelsize=12,
        )
        ax.set_ylabel(
            y_label,
            fontsize=18,
        )
        ax.set_xlabel(
            "Time points",
            fontsize=18,
        )
        ax.set_title(
            "Class Size at Time Points",
            fontsize=20,
        )

        y_max_scaled = base.max() * 1.1 if base.max() > 0 else 1.0

        ax.set_ylim(
            0,
            y_max_scaled,
        )
        ax.yaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=8,
                integer=True,
            ),
        )

        # 6. Add Legend
        handles = [
            patches_by_class[cls]
            for cls in classes_for_legend
            if cls in patches_by_class
        ]
        labels = [
            cls
            for cls in classes_for_legend
            if cls in patches_by_class
        ]

        ax.legend(
            handles,
            labels,
            title="Class",
            title_fontsize=16,
            bbox_to_anchor=(
                1.02,
                0.5,
            ),
            loc="center left",
            frameon=False,
            fontsize=16,
            alignment="left",
        )

        plt.tight_layout()
        plt.show()

        return (
            "Graphic rendered successfully on the user's screen.\n\n"
            "Here is a suggested description to share with the user:\n"
            "----------------------------------------------------------------------\n"
            "The graphic sorts the land cover classes based on their overall net\n"
            "size change. It places the class with the largest gain at the base\n"
            "of the stacked bar. It positions the remaining classes sequentially\n"
            "above it. Finally, it places the class experiencing the largest loss\n"
            "at the very top of the stack.\n"
            "----------------------------------------------------------------------"
        )

### 4.2 Number of changes

In [ ]:
# @title 4.2.1 Number of Changes Processing Module
# @markdown Computes the number of land cover transitions per pixel. Generates dataframes for intervals and overall metrics, alongside methods to render dynamic stacked bar charts directly on the agent interface.

class QuestNumberChanges(QuestBase):
    def compute_number_of_changes(self) -> pd.DataFrame:
        """
        Returns the number of changing pixels in each time interval, split
        by the total number of changes each pixel experienced over the
        whole time series.

        Returns
        -------
        pd.DataFrame
            A dataframe where rows represent time intervals and columns
            represent the total number of changes a pixel experienced.
        """
        cached = self.get_quest_results()
        return cached["tables"]["number_change_per_interval"].copy()

    def compute_overall_changes(self) -> pd.DataFrame:
        """
        Calculates the overall number of changes across the entire study area.

        Returns
        -------
        pd.DataFrame
            A dataframe indexing the number of changes with columns for
            absolute pixel counts and percentage of the total study area.
        """
        df_intervals = self.compute_number_of_changes()
        total_study_area_pixels = self.get_quest_results()["study_area_pixels"]

        results = []

        for col_name in df_intervals.columns:
            n_changes = int(col_name)
            total_transitions = df_intervals[col_name].sum()

            # A pixel that changed N times appears N times across the intervals
            if n_changes > 0:
                unique_pixels = total_transitions / n_changes
            else:
                unique_pixels = 0

            if total_study_area_pixels > 0:
                percentage = (unique_pixels / total_study_area_pixels) * 100.0
            else:
                percentage = 0.0

            results.append(
                {
                    "number_of_changes": n_changes,
                    "absolute_pixels": int(unique_pixels),
                    "percentage": percentage,
                }
            )

        df_overall = pd.DataFrame(results)
        df_overall.set_index("number_of_changes", inplace=True)

        return df_overall

    def plot_interval_changes(
        self,
        start_year: int = None,
        end_year: int = None,
    ) -> str:
        """
        Create a stacked bar chart showing the sequence of changes per interval,
        optionally filtered by a specific year range.

        Parameters
        ----------
        start_year : int, optional
            The starting year of the filter range.
        end_year : int, optional
            The ending year of the filter range.

        Returns
        -------
        str
            Confirmation string indicating the chart was rendered on screen.
        """
        df = self.compute_number_of_changes()

        # Filter DataFrame by custom interval range if provided
        if start_year is not None and end_year is not None:
            filtered_rows = []
            for idx in df.index:
                y_start, y_end = map(int, idx.split("-"))
                if y_start >= start_year and y_end <= end_year:
                    filtered_rows.append(idx)

            df = df.loc[filtered_rows]
            if df.empty:
                return f"No interval data found within the range {start_year}-{end_year}."

        # 1. Determine Unit Scaling
        max_val = df.sum(axis=1).max()

        if max_val >= 1_000_000_000_000:
            factor = 1_000_000_000_000.0
            suffix = " (trillion pixels)"
        elif max_val >= 1_000_000_000:
            factor = 1_000_000_000.0
            suffix = " (billion pixels)"
        elif max_val >= 1_000_000:
            factor = 1_000_000.0
            suffix = " (million pixels)"
        elif max_val >= 1_000:
            factor = 1_000.0
            suffix = " (thousand pixels)"
        else:
            factor = 1.0
            suffix = ""

        df_scaled = df / factor

        fig, ax = plt.subplots(figsize=(14, 6))

        n_cols = len(df.columns)
        cmap = plt.cm.viridis_r

        if n_cols > 1:
            colors = [cmap(i / (n_cols - 1)) for i in range(n_cols)]
        else:
            colors = [cmap(0.5)]

        bottom = pd.Series(0.0, index=df_scaled.index)

        for i, col in reversed(list(enumerate(df.columns))):
            vals = df_scaled[col]

            if vals.sum() == 0:
                label_txt = "_nolegend_"
            else:
                label_txt = f"{col}"

            ax.bar(
                df_scaled.index,
                vals,
                bottom=bottom,
                label=label_txt,
                color=colors[i],
                edgecolor="none",
                linewidth=0.5,
                width=0.9,
            )
            bottom += vals

        y_label_text = f"Change{suffix}"
        ax.set_ylabel(y_label_text, fontsize=18)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5, integer=True))

        title_suffix = f" ({start_year}-{end_year})" if start_year and end_year else ""
        ax.set_title(f"Number of Changes during Time Intervals{title_suffix}", fontsize=20, pad=15)

        labels = ax.get_xticklabels()
        n_labels = len(labels)

        if n_labels <= 6:
            rotation = 0
            ha = "center"
        elif n_labels <= 12:
            rotation = 45
            ha = "right"
        else:
            rotation = 90
            ha = "center"

        plt.setp(labels, rotation=rotation, ha=ha, fontsize=18)
        ax.tick_params(axis="y", rotation=0, labelsize=18)

        handles, labels_leg = ax.get_legend_handles_labels()
        leg = ax.legend(
            handles[::-1],
            labels_leg[::-1],
            title="Changes",
            title_fontsize=16,
            bbox_to_anchor=(1.02, 0.5),
            loc="center left",
            frameon=False,
            fontsize=16,
        )

        for patch in leg.get_patches():
            patch.set_linewidth(0)

        plt.tight_layout()
        plt.show()

        return (
            "The graphic shows the number of changes per time interval. The stacked\n"
            "bars represent the total number of changes those specific pixels\n"
            "experienced over the entire time series."
        )

    def plot_overall_changes(
        self,
        metric: str = "percentage",
    ) -> str:
        """
        Generates a stacked bar chart of the overall number of changes.

        Parameters
        ----------
        metric : str, optional
            The metric for the Y-axis, either 'percentage' or 'pixels'.
            Default is 'percentage'.

        Returns
        -------
        str
            Confirmation string indicating the chart was rendered.
        """
        df_overall = self.compute_overall_changes()

        if metric == "percentage":
            plot_values = df_overall["percentage"].to_dict()
        else:
            plot_values = df_overall["absolute_pixels"].to_dict()

        active_changes = [k for k, v in plot_values.items() if v > 0]

        if not active_changes:
            active_changes = list(plot_values.keys())

        n_colors = len(active_changes)
        cmap = plt.cm.viridis_r

        sorted_changes_desc = sorted(active_changes, reverse=True)

        colors = {
            n: cmap(i / (n_colors - 1)) if n_colors > 1 else cmap(0.5)
            for i, n in enumerate(sorted(active_changes))
        }

        fig, ax = plt.subplots(figsize=(6, 6))

        bottom = 0.0
        for n_change in sorted_changes_desc:
            val = plot_values[n_change]
            if val > 0:
                ax.bar(0, val, bottom=bottom, color=colors[n_change], width=0.4, edgecolor="none")
                bottom += val

        if metric == "percentage":
            ax.set_ylabel("Change (% of study area)", fontsize=16)
            ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=10))
        else:
            max_val = bottom

            if max_val >= 1_000_000_000_000:
                scale_factor = 1_000_000_000_000
                y_label = "Change (trillion pixels)"
            elif max_val >= 1_000_000_000:
                scale_factor = 1_000_000_000
                y_label = "Change (billion pixels)"
            elif max_val >= 1_000_000:
                scale_factor = 1_000_000
                y_label = "Change (million pixels)"
            elif max_val >= 1_000:
                scale_factor = 1_000
                y_label = "Change (thousand pixels)"
            elif max_val >= 100:
                scale_factor = 100
                y_label = "Change (hundred pixels)"
            else:
                scale_factor = 1
                y_label = "Change (pixels)"

            ax.set_ylabel(y_label, fontsize=16)
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"{x / scale_factor:g}"))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))

        ax.set_title("Number of Changes Overall", fontsize=18, pad=15)

        for spine in ["top", "right", "bottom", "left"]:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_color("black")
            ax.spines[spine].set_linewidth(0.5)

        ax.tick_params(axis="y", which="major", labelsize=18)
        ax.set_xticks([])

        max_y = bottom * 1.05 if bottom > 0 else 1.0
        ax.set_ylim(0, max_y)

        legend_elements = []

        for n in sorted(active_changes):
            legend_elements.append(
                plt.Rectangle((0, 0), 1, 1, fc=colors[n], label=str(n))
            )

        ax.legend(
            handles=legend_elements,
            title="Changes",
            title_fontsize=16,
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            fontsize=14,
            frameon=False,
        )

        fig.subplots_adjust(left=0.15, right=0.75, bottom=0.1, top=0.9)

        plt.show()

        return (
            "The graphic shows the total changes in the time series, summing\n"
            "all the changes in the time intervals."
        )


class QuestCustomExtentChanges(QuestNumberChanges):
    def __init__(
        self,
        config: dict,
        start_year: int,
        end_year: int,
    ):
        super().__init__(config)
        self.restrict_years(
            start_year=start_year,
            end_year=end_year,
        )

### 4.3 Trajectory Analysis

In [ ]:
# @title 4.3.1 Trajectory Processing Module
# @markdown Reads the pixel trajectories computed by the QUEST core engine, which classifies each pixel by land cover persistence and transitions inside the common valid mask, and includes plotting methods.

class QuestTrajectory(QuestBase):
    def compute_trajectory_intervals(self) -> pd.DataFrame:
        """
        Calculates the number of trajectory changes in each time interval.

        Returns
        -------
        pd.DataFrame
            A dataframe with time intervals as index and trajectory
            classes (2, 3, 4, 5) as columns.
        """
        cached = self.get_quest_results()
        return cached["tables"]["trajectory_contributions_per_interval"].copy()

    def compute_overall_trajectories(self) -> pd.DataFrame:
        """
        Calculates the total unique pixels per trajectory class across the
        entire study area.

        Returns
        -------
        pd.DataFrame
            A dataframe indexing the trajectory class (2, 3, 4, 5) with
            columns for absolute pixel counts and percentage.
        """
        cached = self.get_quest_results()
        trajectory_counts = cached["results"]["trajectory_counts"]
        total_study_area_pixels = cached["study_area_pixels"]

        results = []
        for c in [2, 3, 4, 5]:
            unique_pixels = int(trajectory_counts[c])

            if total_study_area_pixels > 0:
                percentage = (unique_pixels / total_study_area_pixels) * 100.0
            else:
                percentage = 0.0

            results.append(
                {
                    "Trajectory_Class": c,
                    "absolute_pixels": unique_pixels,
                    "percentage": percentage,
                }
            )

        df_overall = pd.DataFrame(results)
        df_overall.set_index("Trajectory_Class", inplace=True)

        return df_overall

    def plot_trajectory_intervals(
        self,
        start_year: int = None,
        end_year: int = None,
    ) -> str:
        """
        Create a stacked bar chart for trajectory contributions per interval,
        optionally filtered by a specific year range.

        Parameters
        ----------
        start_year : int, optional
            The starting year of the filter range.
        end_year : int, optional
            The ending year of the filter range.

        Returns
        -------
        str
            Confirmation string indicating the chart was rendered on screen.
        """
        df = self.compute_trajectory_intervals()
        df.columns = df.columns.astype(int)

        if start_year is not None and end_year is not None:
            filtered_rows = []
            for idx in df.index:
                y_start, y_end = map(int, idx.split("-"))
                if y_start >= start_year and y_end <= end_year:
                    filtered_rows.append(idx)

            df = df.loc[filtered_rows]
            if df.empty:
                return f"No interval data found within the range {start_year}-{end_year}."

        max_val = df.sum(axis=1).max()

        if max_val >= 1_000_000_000_000:
            scale_factor = 1_000_000_000_000.0
            y_label = "Change (trillion pixels)"
        elif max_val >= 1_000_000_000:
            scale_factor = 1_000_000_000.0
            y_label = "Change (billion pixels)"
        elif max_val >= 1_000_000:
            scale_factor = 1_000_000.0
            y_label = "Change (million pixels)"
        elif max_val >= 1_000:
            scale_factor = 1_000.0
            y_label = "Change (thousand pixels)"
        else:
            scale_factor = 1.0
            y_label = "Change (pixels)"

        df_scaled = df / scale_factor

        colors = {
            2: "#990033",
            3: "#FDE724",
            4: "#ff9900",
            5: "#000066",
        }

        stack_order = [5, 4, 3, 2]

        fig, ax = plt.subplots(figsize=(12, 6))

        bottom = pd.Series(0.0, index=df_scaled.index)

        for traj_id in stack_order:
            if traj_id in df_scaled.columns:
                values = df_scaled[traj_id]
                ax.bar(df_scaled.index, values, label=str(traj_id), bottom=bottom, color=colors[traj_id], edgecolor="none", width=0.9)
                bottom += values

        ax.set_ylabel(y_label, fontsize=18)

        title_suffix = f" ({start_year}-{end_year})" if start_year and end_year else ""
        ax.set_title(f"Trajectories during Time Intervals{title_suffix}", fontsize=20, pad=15)

        labels = ax.get_xticklabels()
        n_labels = len(labels)

        if n_labels <= 6:
            rotation = 0
            ha = "center"
        elif n_labels <= 12:
            rotation = 45
            ha = "right"
        else:
            rotation = 90
            ha = "center"

        plt.setp(labels, rotation=rotation, ha=ha, fontsize=18)
        ax.tick_params(axis="y", labelsize=18)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=5))

        for spine in ["top", "right", "left", "bottom"]:
            ax.spines[spine].set_visible(True)

        handles, labels_leg = ax.get_legend_handles_labels()
        if handles:
            legend_order_map = {"2": 0, "3": 1, "4": 2, "5": 3}
            sorted_pairs = sorted(zip(handles, labels_leg), key=lambda x: legend_order_map.get(x[1], 99))
            sorted_handles, sorted_labels = zip(*sorted_pairs)

            ax.legend(
                sorted_handles,
                sorted_labels,
                loc="center left",
                bbox_to_anchor=(1.01, 0.5),
                title="Trajectory",
                title_fontsize=14,
                alignment="left",
                fontsize=14,
                frameon=False,
            )

        plt.tight_layout()
        plt.show()

        return "Trajectory interval chart rendered successfully."

    def plot_overall_trajectories(
        self,
        metric: str = "percentage",
    ) -> str:
        """
        Generates a stacked bar chart of the overall trajectory distributions.

        Parameters
        ----------
        metric : str, optional
            The metric for the Y-axis, either 'percentage' or 'pixels'.
            Default is 'percentage'.

        Returns
        -------
        str
            Confirmation string indicating the chart was rendered.
        """
        df_overall = self.compute_overall_trajectories()

        if metric == "percentage":
            plot_values = df_overall["percentage"].to_dict()
        else:
            plot_values = df_overall["absolute_pixels"].to_dict()

        ordered_trajs = [5, 4, 3, 2]

        colors = {
            2: "#990033",
            3: "#FDE724",
            4: "#ff9900",
            5: "#000066",
        }

        fig, ax = plt.subplots(figsize=(6, 6))

        bottom = 0.0
        for traj in ordered_trajs:
            val = plot_values.get(traj, 0.0)
            ax.bar(0, val, bottom=bottom, color=colors[traj], width=0.4, edgecolor="none")
            bottom += val

        if metric == "percentage":
            ax.set_ylabel("Change (% of study area)", fontsize=16)
            ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=10))
        else:
            max_val = bottom

            if max_val >= 1_000_000_000_000:
                scale_factor = 1_000_000_000_000
                y_label = "Change (trillion pixels)"
            elif max_val >= 1_000_000_000:
                scale_factor = 1_000_000_000
                y_label = "Change (billion pixels)"
            elif max_val >= 1_000_000:
                scale_factor = 1_000_000
                y_label = "Change (million pixels)"
            elif max_val >= 1_000:
                scale_factor = 1_000
                y_label = "Change (thousand pixels)"
            elif max_val >= 100:
                scale_factor = 100
                y_label = "Change (hundred pixels)"
            else:
                scale_factor = 1
                y_label = "Change (pixels)"

            ax.set_ylabel(y_label, fontsize=16)
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"{x / scale_factor:g}"))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))

        ax.set_title("Trajectories Overall", fontsize=18, pad=15)

        for spine in ["top", "right", "bottom", "left"]:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_color("black")
            ax.spines[spine].set_linewidth(0.5)

        ax.tick_params(axis="y", which="major", labelsize=18)
        ax.set_xticks([])
        ax.set_ylim(0, bottom * 1.05 if bottom > 0 else 1.0)

        legend_elements = [
            plt.Rectangle((0, 0), 1, 1, fc=colors[2], label="2"),
            plt.Rectangle((0, 0), 1, 1, fc=colors[3], label="3"),
            plt.Rectangle((0, 0), 1, 1, fc=colors[4], label="4"),
            plt.Rectangle((0, 0), 1, 1, fc=colors[5], label="5"),
        ]

        ax.legend(
            handles=legend_elements,
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            title="Trajectory",
            title_fontsize=14,
            alignment="left",
            fontsize=14,
            frameon=False,
        )

        fig.subplots_adjust(left=0.15, right=0.75, bottom=0.1, top=0.9)

        plt.show()

        return f"Overall trajectory chart ({metric}) rendered successfully."


class QuestCustomExtentTrajectory(QuestTrajectory):
    def __init__(
        self,
        config: dict,
        start_year: int,
        end_year: int,
    ):
        super().__init__(config)
        self.restrict_years(
            start_year=start_year,
            end_year=end_year,
        )

### 4.4 Transition matrices

In [ ]:
# @title 4.4.1 Transition Matrix Processing Module
# @markdown Reads the interval, aggregated and trajectory transition matrices computed by the QUEST core engine and plots them as heatmaps.

class QuestTransition(QuestBase):
    def compute_transition_matrices(
        self,
    ) -> dict[str, pd.DataFrame]:
        """
        Return the interval, aggregated and trajectory transition matrices.

        Returns
        -------
        dict[str, pd.DataFrame]
            Keys "time_intervals", "extent", "allocation_exchange",
            "quantity_allocation_shift", "alternation_exchange",
            "alternation_shift", "indirect_extent", "trajectory_1" to
            "trajectory_5", and "interval_<start>-<end>" for each interval.
            The five trajectory matrices sum to "time_intervals".
        """
        tables = self.get_quest_results()["tables"]

        output_dfs = {
            name: df.copy()
            for name, df in tables["aggregated_matrices"].items()
            if name != "indirect_ratio"
        }

        for traj, df in tables["trajectory_matrices"].items():
            output_dfs[f"trajectory_{traj}"] = df.copy()

        for interval_name, df in tables["interval_matrices"].items():
            output_dfs[f"interval_{interval_name}"] = df.copy()

        return output_dfs

    def _compute_net_change(
        self,
        df_sum: pd.DataFrame,
    ) -> pd.Series:
        """
        Compute net change per class from the time_intervals matrix.

        Parameters
        ----------
        df_sum : pd.DataFrame
            Square transition matrix representing total transitions over
            the full time span.

        Returns
        -------
        pd.Series
            Net change for each class (gains minus losses).

        Examples
        --------
        >>> gains_losses = self._compute_net_change(df_sum=matrix)
        """
        m_vals = df_sum.values.astype(float).copy()
        np.fill_diagonal(m_vals, 0.0)
        gains = m_vals.sum(axis=0)
        losses = m_vals.sum(axis=1)
        net_change = gains - losses
        return pd.Series(net_change, index=df_sum.index)

    def _reorder_matrices(
        self,
        matrices_dict: dict[str, pd.DataFrame],
    ) -> dict[str, pd.DataFrame]:
        """
        Reorder rows and columns of all matrices using net change.

        Parameters
        ----------
        matrices_dict : dict[str, pd.DataFrame]
            Dictionary containing all loaded DataFrames.

        Returns
        -------
        dict[str, pd.DataFrame]
            Dictionary with all input DataFrames reindexed based on net change.

        Examples
        --------
        >>> reordered_mats = self._reorder_matrices(matrices_dict=mats)
        """
        net_change = self._compute_net_change(df_sum=matrices_dict["time_intervals"])
        order_labels = net_change.sort_values(ascending=True).index.tolist()

        reordered = {}
        for k, df in matrices_dict.items():
            reordered[k] = df.reindex(index=order_labels, columns=order_labels)

        return reordered

    @staticmethod
    def _annotate_heatmap(
        ax: plt.Axes,
        m_vals: np.ndarray,
        fontsize: int = 11,
        show_diagonal: bool = True,
        equalize_diagonal_font: bool = False,
    ) -> None:
        """
        Annotate a heatmap with integer cell values using a standardized font size and adaptive text contrast.
        """
        if m_vals.size == 0:
            return

        rows, cols = m_vals.shape
        max_val_str = str(int(np.nanmax(np.abs(m_vals))))
        max_len = len(max_val_str)
        uniform_fontsize = 9.5 if max_len > 6 else 11

        m_off = m_vals.copy()
        np.fill_diagonal(m_off, np.nan)
        max_off = np.nanmax(m_off) if np.any(np.isfinite(m_off)) else 1.0
        threshold = 0.5 * max_off

        for i in range(rows):
            for j in range(cols):
                val = float(m_vals[i, j])

                if i == j:
                    if not show_diagonal:
                        continue
                    color = "white"
                else:
                    if val >= threshold and val > 0:
                        color = "white"
                    else:
                        color = "black"

                txt = f"{int(round(val))}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=uniform_fontsize, color=color, clip_on=True)

    def plot_matrix_heatmap(
        self,
        matrix_key: str,
        class_labels_dict: dict = None,
    ) -> str:
        """
        Plot a square transition matrix as a heatmap.
        """
        matrices = self.compute_transition_matrices()
        matrices = self._reorder_matrices(matrices_dict=matrices)

        if matrix_key not in matrices:
            return f"Matrix '{matrix_key}' not found. Available keys: {list(matrices.keys())}"

        df = matrices[matrix_key]

        if class_labels_dict is None:
            class_labels_dict = {}

        start_year = self.years[0]
        end_year = self.years[-1]

        matrix_meta = {
            "time_intervals": ("Time Intervals", "flow"),
            "extent": ("Extent", "stock"),
            "allocation_exchange": ("Allocation Exchange", "stock"),
            "quantity_allocation_shift": ("Quantity & Allocation Shift", "stock"),
            "alternation_exchange": ("Alternation Exchange", "flow"),
            "alternation_shift": ("Alternation Shift", "flow"),
            "indirect_extent": ("Indirect", "stock"),
            "trajectory_1": ("Trajectory 1", "flow"),
            "trajectory_2": ("Trajectory 2", "flow"),
            "trajectory_3": ("Trajectory 3", "flow"),
            "trajectory_4": ("Trajectory 4", "flow"),
            "trajectory_5": ("Trajectory 5", "flow"),
        }

        if matrix_key.startswith("interval_"):
            title_part = "Time Interval"
            m_type = "flow"
            interval_str = matrix_key.replace("interval_", "")
            full_title = f"{title_part} {interval_str}"
        else:
            title_part, m_type = matrix_meta.get(matrix_key, ("Custom Matrix", "flow"))
            label_suffix = "..." if m_type == "flow" else "-"
            full_title = f"{title_part} {start_year}{label_suffix}{end_year}"

        is_flow = (m_type == "flow")
        xlabel = "To class" if is_flow else "End class"
        ylabel = "From class" if is_flow else "Begin class"
        cbar_label = "Transitions" if is_flow else "Pixels"
        is_extent = (matrix_key == "extent")

        labels = list(df.index)
        tick_names = [self.class_labels_dict.get(int(lab), {}).get("name", str(lab)) for lab in labels]

        matrix_values = df.values.astype(float)

        matrix_scale = matrix_values.copy()
        np.fill_diagonal(matrix_scale, 0.0)
        finite_vals_raw = matrix_scale[np.isfinite(matrix_scale)]

        max_abs = float(np.nanmax(np.abs(finite_vals_raw))) if finite_vals_raw.size > 0 else 0.0

        if max_abs >= 1_000_000_000_000:
            factor = 1_000_000_000_000.0
            suffix = "T"
        elif max_abs >= 1_000_000_000:
            factor = 1_000_000_000.0
            suffix = "B"
        elif max_abs >= 1_000_000:
            factor = 1_000_000.0
            suffix = "M"
        elif max_abs >= 1_000:
            factor = 1_000.0
            suffix = "k"
        else:
            factor = 1.0
            suffix = ""

        matrix_display_values = matrix_values / factor

        matrix_scale_disp = matrix_display_values.copy()
        np.fill_diagonal(matrix_scale_disp, 0.0)
        finite_vals_disp = matrix_scale_disp[np.isfinite(matrix_scale_disp)]

        has_negative = False
        vmin_eff, vmax_eff = 0.0, 1.0

        if finite_vals_disp.size > 0:
            has_negative = float(np.nanmin(finite_vals_disp)) < 0.0
            min_val = float(np.nanmin(finite_vals_disp))
            max_val = float(np.nanmax(finite_vals_disp))

            if has_negative:
                vmin_eff, vmax_eff = min_val, max_val
            else:
                vmin_eff = 0.0
                vmax_eff = float(max_val)

            if vmin_eff == vmax_eff:
                vmax_eff += 1.0

        if vmax_eff <= 10:
            ceiling_scaled = np.ceil(vmax_eff)
        else:
            ceiling_scaled = np.ceil(vmax_eff / 5.0) * 5.0
        vmax_eff = ceiling_scaled

        nrows, ncols = df.shape
        figsize = (max(7.0, 1.2 * ncols), max(6.0, 1.2 * nrows))

        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

        cmap_neg = mcolors.LinearSegmentedColormap.from_list("CustomBlues", ["#08306b", "#b3e0ff"])

        if has_negative:
            matrix_pos = np.ma.masked_less_equal(matrix_display_values, 0.0)
            norm_pos = mcolors.Normalize(vmin=0.0, vmax=vmax_eff)
            ax.imshow(matrix_pos, aspect="equal", cmap=plt.cm.YlOrRd, norm=norm_pos)

            matrix_neg = np.ma.masked_where(matrix_display_values >= 0.0, matrix_display_values)
            norm_neg = mcolors.Normalize(vmin=vmin_eff, vmax=0.0)
            ax.imshow(matrix_neg, aspect="equal", cmap=cmap_neg, norm=norm_neg)
        else:
            matrix_pos = np.ma.masked_equal(matrix_display_values, 0.0)
            norm_pos = mcolors.Normalize(vmin=vmin_eff, vmax=vmax_eff)
            ax.imshow(matrix_pos, aspect="equal", cmap=plt.cm.YlOrRd, norm=norm_pos)

        diag_mask = np.eye(nrows, dtype=bool)
        matrix_diag = np.ma.masked_where(~diag_mask, np.ones_like(matrix_display_values))
        ax.imshow(matrix_diag, aspect="equal", cmap=mcolors.ListedColormap(["black"]), vmin=0.0, vmax=1.0)

        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(tick_names, rotation=90, fontsize=12)
        ax.set_yticklabels(tick_names, fontsize=12)
        ax.set_xlabel(xlabel, fontsize=14)
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_title(full_title, fontsize=18)

        n_bar = 256
        vals = np.linspace(vmin_eff, vmax_eff, n_bar)
        colors_bar = []

        for v in vals:
            if has_negative and v < 0.0:
                t = (v - vmin_eff) / (0.0 - vmin_eff) if vmin_eff < 0 else 0
                colors_bar.append(cmap_neg(t))
            elif v == 0:
                colors_bar.append((1.0, 1.0, 1.0, 1.0))
            else:
                t = max(0.0, v) / vmax_eff if vmax_eff > 0.0 else 0.0
                colors_bar.append(plt.cm.YlOrRd(t))

        cmap_bar = mcolors.ListedColormap(colors_bar)
        norm_bar = mcolors.Normalize(vmin=vmin_eff, vmax=vmax_eff)

        scalar_mappable = plt.cm.ScalarMappable(cmap=cmap_bar, norm=norm_bar)
        scalar_mappable.set_array([])

        cbar = fig.colorbar(scalar_mappable, ax=ax, fraction=0.025, pad=0.02)

        cbar.locator = mticker.MaxNLocator(nbins=4, integer=True, prune="both")
        cbar.formatter = mticker.FuncFormatter(lambda x, pos: f"{int(x)}")
        cbar.update_ticks()

        mapping = {
            "": cbar_label,
            "k": f"{cbar_label} (thousands)",
            "M": f"{cbar_label} (millions)",
            "B": f"{cbar_label} (billions)",
            "T": f"{cbar_label} (trillions)",
        }

        cbar.set_label(mapping.get(suffix, cbar_label), fontsize=12)

        plt.show()
        return f"Heatmap for {matrix_key} rendered successfully."

### 4.5 Change Components

In [ ]:
# @title 4.5.1 Change Components Processing Module
# @markdown Houses the computational engine to decompose transition matrices into Quantity, Exchange, and Shift components, structuring the results into a standardized master table.

class QuestChangeComponents(QuestTransition):
    """
    Calculates and organizes change components from transition matrices,
    differentiating strictly between Allocation and Alternation logic.
    """

    def compute_change_components_table(self) -> pd.DataFrame:
        """
        Return the change components of every class for every interval and
        for the extent, alternation exchange and alternation shift matrices.

        Alternation Shift excludes the Indirect Extent, as in the
        03_quest_pie notebook.

        Returns
        -------
        pd.DataFrame
            Columns Time_Interval, Class, Component, Gain and Loss.

        Examples
        --------
        >>> engine = QuestChangeComponents(config=QUEST_CONFIG)
        >>> df = engine.compute_change_components_table()
        """
        return self.get_quest_results()["components"].copy()

    def plot_change_components_intervals(
        self,
    ) -> str:
        """
        Plot the aggregated change components across individual time intervals.

        Returns
        -------
        str
            A success message indicating the chart was rendered.

        Examples
        --------
        >>> engine = QuestChangeComponents(config=QUEST_CONFIG)
        >>> engine.plot_change_components_intervals()
        """
        df = self.compute_change_components_table()

        time_df = df[
            df["Time_Interval"].str.contains(
                "-",
                na=False,
            )
        ]

        if time_df.empty:
            return "No interval data found containing '-' in the nomenclature."

        totals = (
            time_df.groupby(
                [
                    "Time_Interval",
                    "Component",
                ]
            )["Gain"]
            .sum()
            .unstack(
                fill_value=0.0,
            )
        )

        for col in [
            "Quantity",
            "Shift",
            "Exchange",
        ]:
            if col not in totals.columns:
                totals[col] = 0.0

        max_val = float(
            totals[
                [
                    "Quantity",
                    "Shift",
                    "Exchange",
                ]
            ].to_numpy().max()
        )

        if max_val >= 1_000_000_000_000:
            scale_factor = 1_000_000_000_000.0
            y_label = "Change (trillion pixels)"
        elif max_val >= 1_000_000_000:
            scale_factor = 1_000_000_000.0
            y_label = "Change (billion pixels)"
        elif max_val >= 1_000_000:
            scale_factor = 1_000_000.0
            y_label = "Change (million pixels)"
        elif max_val >= 1_000:
            scale_factor = 1_000.0
            y_label = "Change (thousand pixels)"
        elif max_val >= 100:
            scale_factor = 100.0
            y_label = "Change (hundred pixels)"
        else:
            scale_factor = 1.0
            y_label = "Change (pixels)"

        scaled_totals = totals[
            [
                "Quantity",
                "Shift",
                "Exchange",
            ]
        ] / scale_factor

        stacked_max = float(
            scaled_totals.sum(
                axis=1,
            ).max()
        )

        fig, ax = plt.subplots(
            figsize=(
                14,
                6,
            ),
        )

        colors = [
            "#1f77b4",
            "#2ca02c",
            "#ffd700",
        ]

        components = [
            "Quantity",
            "Shift",
            "Exchange",
        ]

        for idx, comp in enumerate(
            components,
        ):
            bottom_values = (
                scaled_totals.iloc[:, :idx].sum(
                    axis=1,
                )
                if idx > 0
                else np.zeros(
                    len(
                        totals,
                    ),
                )
            )
            ax.bar(
                totals.index,
                scaled_totals[comp],
                label=comp,
                color=colors[idx],
                edgecolor="none",
                bottom=bottom_values,
                width=0.9,
            )

        ax.set_ylabel(
            y_label,
            fontsize=18,
        )
        ax.set_title(
            "Change Components during Time Intervals",
            fontsize=20,
            pad=15,
        )
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=18,
        )

        labels = ax.get_xticklabels()
        n_labels = len(
            labels,
        )

        if n_labels <= 6:
            rotation = 0
            ha = "center"
        elif n_labels <= 12:
            rotation = 45
            ha = "right"
        else:
            rotation = 90
            ha = "center"

        plt.setp(
            labels,
            rotation=rotation,
            ha=ha,
        )

        y_max_scaled = stacked_max * 1.1 if stacked_max > 0 else 1.0

        ax.set_ylim(
            0,
            y_max_scaled,
        )
        ax.yaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=6,
                integer=True,
            ),
        )

        legend_elements = [
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color="#ffd700",
                label="Allocation Exchange",
            ),
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color="#2ca02c",
                label="Allocation Shift",
            ),
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color="#1f77b4",
                label="Quantity",
            ),
        ]

        ax.legend(
            handles=legend_elements,
            loc="center left",
            bbox_to_anchor=(
                1.01,
                0.5,
            ),
            title="Component",
            title_fontsize=16,
            fontsize=16,
            alignment="left",
            frameon=False,
        )

        plt.tight_layout()
        plt.show()

        return "Time intervals components chart rendered successfully."

    def plot_change_components_overall(
        self,
    ) -> str:
        """
        Plot overall change components as a single stacked bar with auto-scaled y-axis.

        Returns
        -------
        str
            A success message indicating the chart was rendered.

        Examples
        --------
        >>> engine.plot_change_components_overall()
        """
        # 1. Load standardized data table from memory
        df = self.compute_change_components_table()

        # 2. Define colors and component stacking order
        components_color = {
            "Quantity": "#1f77b4",
            "Allocation_Exchange": "#ffd700",
            "Alternation_Exchange": "#ff8080",
            "Allocation_Shift": "#2ca02c",
            "Alternation_Shift": "#990099",
        }

        component_order = [
            "Quantity",
            "Allocation_Shift",
            "Allocation_Exchange",
            "Alternation_Shift",
            "Alternation_Exchange",
        ]

        # 3. Aggregate totals per component with the core engine. Alternation
        #    Shift already excludes the Indirect Extent.
        component_totals = (
            summarize_change_components(
                df,
            )
            .set_index(
                "Component",
            )["Pixels"]
            .astype(
                float,
            )
            .to_dict()
        )

        # 4. Automatic scale based on the sum of all stacked components
        total_change = sum(
            component_totals.values(),
        )

        if total_change >= 1_000_000_000_000:
            scale_factor = 1_000_000_000_000.0
            y_label = "Change (trillion pixels)"
        elif total_change >= 1_000_000_000:
            scale_factor = 1_000_000_000.0
            y_label = "Change (billion pixels)"
        elif total_change >= 1_000_000:
            scale_factor = 1_000_000.0
            y_label = "Change (million pixels)"
        elif total_change >= 1_000:
            scale_factor = 1_000.0
            y_label = "Change (thousand pixels)"
        elif total_change >= 100:
            scale_factor = 100.0
            y_label = "Change (hundred pixels)"
        else:
            scale_factor = 1.0
            y_label = "Change (pixels)"

        # 5. Initialize figure and axis
        fig, ax = plt.subplots(
            figsize=(
                6,
                6,
            ),
        )

        # 6. Plot each component in a stacked bar at a single x-position
        bottom = 0.0
        for component in component_order:
            value = (
                component_totals.get(
                    component,
                    0.0,
                )
                / scale_factor
            )
            ax.bar(
                x=0,
                height=value,
                bottom=bottom,
                color=components_color[
                    component
                ],
                edgecolor="none",
                width=0.4,
            )
            bottom += value

        # 7. Axes formatting and labels
        ax.set_ylabel(
            y_label,
            fontsize=16,
        )
        ax.set_title(
            "Change Components Overall",
            fontsize=18,
            pad=15,
        )
        ax.set_xticks(
            [],
        )

        ax.tick_params(
            axis="both",
            which="major",
            labelsize=18,
        )

        # 8. Set y-axis limits and major tick locators
        y_max_scaled = bottom * 1.05 if bottom > 0 else 1.0
        ax.set_ylim(
            0,
            y_max_scaled,
        )

        ax.yaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=10,
                integer=True,
            ),
        )

        ax.yaxis.set_major_formatter(
            mticker.FormatStrFormatter(
                "%d",
            ),
        )

        # 9. Configure visible spines for the plot frame
        for spine in [
            "top",
            "right",
            "bottom",
            "left",
        ]:
            ax.spines[
                spine
            ].set_visible(
                True,
            )
            ax.spines[
                spine
            ].set_color(
                "black",
            )
            ax.spines[
                spine
            ].set_linewidth(
                0.5,
            )

        # 10. Define custom legend elements matching the components
        legend_elements = [
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color=components_color[
                    "Alternation_Exchange"
                ],
                label="Alternation Exchange",
            ),
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color=components_color[
                    "Alternation_Shift"
                ],
                label="Alternation Shift",
            ),
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color=components_color[
                    "Allocation_Exchange"
                ],
                label="Allocation Exchange",
            ),
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color=components_color[
                    "Allocation_Shift"
                ],
                label="Allocation Shift",
            ),
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color=components_color[
                    "Quantity"
                ],
                label="Quantity",
            ),
        ]

        ax.legend(
            handles=legend_elements,
            loc="center left",
            bbox_to_anchor=(
                1.05,
                0.5,
            ),
            title="Component",
            title_fontsize=14,
            fontsize=14,
            alignment="left",
            frameon=False,
        )

        fig.subplots_adjust(
            left=0.15,
            right=0.75,
            bottom=0.1,
            top=0.9,
        )

        plt.show()

        return "Overall change components chart rendered successfully."

    def plot_change_components_by_class(
        self,
    ) -> str:
        """
        Plot per-class gains and losses as stacked bars with auto-scaled y-axis.

        Returns
        -------
        str
            A success message indicating the chart was rendered.

        Examples
        --------
        >>> engine.plot_change_components_by_class()
        """
        # 1. Load standardized data table from memory
        df = self.compute_change_components_table()

        # 2. Filter target intervals and adjust quantity label
        target_intervals = [
            "extent",
            "alternation_shift",
            "alternation_exchange",
        ]

        df_sub = df[
            df["Time_Interval"].isin(
                target_intervals,
            )
        ].copy()

        df_sub["Component"] = df_sub["Component"].str.replace(
            "Allocation_Quantity",
            "Quantity",
        )

        # 3. Define colors and component groups
        components_map = {
            "Quantity": "#1f77b4",
            "Allocation_Exchange": "#ffd700",
            "Alternation_Exchange": "#ff8080",
            "Allocation_Shift": "#2ca02c",
            "Alternation_Shift": "#990099",
        }

        comp_groups = [
            "Quantity",
            "Allocation_Shift",
            "Allocation_Exchange",
        ]

        # 4. Identify classes and sort by net quantity change
        classes = sorted(
            df_sub["Class"].unique(),
        )

        class_stats = []
        for cls in classes:
            c_data = df_sub[
                df_sub["Class"] == cls
            ]
            qty_gain = float(
                c_data[
                    c_data["Component"] == "Quantity"
                ]["Gain"].sum()
            )
            qty_loss = float(
                c_data[
                    c_data["Component"] == "Quantity"
                ]["Loss"].sum()
            )

            class_stats.append(
                (
                    cls,
                    qty_gain - qty_loss,
                ),
            )

        ordered_classes = [
            x[0]
            for x in sorted(
                class_stats,
                key=lambda x: x[1],
            )
        ]

        # 5. Prepare plot data handling the net logic for alternation
        plot_data = []
        max_val = 0.0

        for cls in ordered_classes:
            c_data = df_sub[
                df_sub["Class"] == cls
            ]

            gains = {}
            for comp in comp_groups:
                gains[comp] = float(
                    c_data[
                        c_data["Component"] == comp
                    ]["Gain"].sum()
                )

            raw_alt_exc = float(
                c_data[
                    c_data["Component"] == "Alternation_Exchange"
                ]["Gain"].sum()
            )

            raw_alt_shift = float(
                c_data[
                    c_data["Component"] == "Alternation_Shift"
                ]["Gain"].sum()
            )

            net_alt = raw_alt_exc + raw_alt_shift

            adj_alt_exc = 0.0
            adj_alt_shift = 0.0

            if net_alt > 0.0001:
                if raw_alt_exc > 0.0:
                    adj_alt_exc = raw_alt_exc
                    adj_alt_shift = max(
                        0.0,
                        net_alt - raw_alt_exc,
                    )
                else:
                    adj_alt_exc = 0.0
                    adj_alt_shift = net_alt

            gains["Alternation_Exchange"] = adj_alt_exc
            gains["Alternation_Shift"] = adj_alt_shift

            losses = {}
            for comp in comp_groups:
                losses[comp] = float(
                    c_data[
                        c_data["Component"] == comp
                    ]["Loss"].sum()
                )

            raw_alt_exc_l = float(
                c_data[
                    c_data["Component"] == "Alternation_Exchange"
                ]["Loss"].sum()
            )

            raw_alt_shift_l = float(
                c_data[
                    c_data["Component"] == "Alternation_Shift"
                ]["Loss"].sum()
            )

            net_alt_l = raw_alt_exc_l + raw_alt_shift_l

            adj_alt_exc_l = 0.0
            adj_alt_shift_l = 0.0

            if net_alt_l > 0.0001:
                if raw_alt_exc_l > 0.0:
                    adj_alt_exc_l = raw_alt_exc_l
                    adj_alt_shift_l = max(
                        0.0,
                        net_alt_l - raw_alt_exc_l,
                    )
                else:
                    adj_alt_exc_l = 0.0
                    adj_alt_shift_l = net_alt_l

            losses["Alternation_Exchange"] = adj_alt_exc_l
            losses["Alternation_Shift"] = adj_alt_shift_l

            total_g = sum(
                gains.values(),
            )
            total_l = sum(
                losses.values(),
            )
            max_val = max(
                max_val,
                total_g,
                total_l,
            )

            plot_data.append(
                {
                    "class": cls,
                    "gains": gains,
                    "losses": losses,
                },
            )

        # 6. Determine scale factor
        if max_val >= 1_000_000_000_000:
            scale_factor = 1_000_000_000_000.0
            y_label = "Loss and Gain (trillion pixels)"
        elif max_val >= 1_000_000_000:
            scale_factor = 1_000_000_000.0
            y_label = "Loss and Gain (billion pixels)"
        elif max_val >= 1_000_000:
            scale_factor = 1_000_000.0
            y_label = "Loss and Gain (million pixels)"
        elif max_val >= 1_000:
            scale_factor = 1_000.0
            y_label = "Loss and Gain (thousand pixels)"
        elif max_val >= 100:
            scale_factor = 100.0
            y_label = "Loss and Gain (hundred pixels)"
        else:
            scale_factor = 1.0
            y_label = "Loss and Gain (pixels)"

        # 7. Generate figure and plot bars
        fig, ax = plt.subplots(
            figsize=(
                14,
                8,
            ),
        )
        fig.subplots_adjust(
            left=0.1,
            right=0.75,
        )

        x_pos = np.arange(
            len(
                ordered_classes,
            ),
        )
        width = 0.6

        stack_order = [
            "Quantity",
            "Allocation_Shift",
            "Allocation_Exchange",
            "Alternation_Shift",
            "Alternation_Exchange",
        ]

        for idx, item in enumerate(
            plot_data,
        ):
            bottom_g = 0.0
            for comp in stack_order:
                val = item["gains"][comp] / scale_factor
                if val > 0.0:
                    ax.bar(
                        x_pos[idx],
                        val,
                        width,
                        bottom=bottom_g,
                        color=components_map[
                            comp
                        ],
                        edgecolor="none",
                    )
                    bottom_g += val

            bottom_l = 0.0
            for comp in stack_order:
                val = item["losses"][comp] / scale_factor
                if val > 0.0:
                    ax.bar(
                        x_pos[idx],
                        -val,
                        width,
                        bottom=bottom_l,
                        color=components_map[
                            comp
                        ],
                        edgecolor="none",
                    )
                    bottom_l -= val

        # 8. Format axes and text
        ax.set_xticks(
            x_pos,
        )
        ax.set_xticklabels(
            ordered_classes,
            rotation=0,
            ha="center",
            fontsize=14,
        )
        ax.axhline(
            0,
            color="black",
            linewidth=0.8,
        )

        ax.set_ylabel(
            y_label,
            fontsize=20,
        )

        ax.set_title(
            "Change Components by Class",
            fontsize=20,
            pad=15,
        )

        ax.tick_params(
            axis="x",
            labelsize=16,
        )

        ax.tick_params(
            axis="y",
            labelsize=24,
        )

        limit = (max_val / scale_factor) * 1.1 if max_val > 0 else 1.0
        ax.set_ylim(
            -limit,
            limit,
        )
        ax.yaxis.set_major_locator(
            mticker.MaxNLocator(
                integer=True,
                nbins=10,
            ),
        )
        ax.yaxis.set_major_formatter(
            mticker.FormatStrFormatter(
                "%d",
            ),
        )

        # 9. Create custom legend
        handles = [
            plt.Rectangle(
                (0, 0),
                1,
                1,
                color=components_map[
                    c
                ],
                label=c.replace(
                    "_",
                    " ",
                ),
            )
            for c in reversed(
                stack_order,
            )
        ]

        ax.legend(
            handles=handles,
            loc="center left",
            bbox_to_anchor=(
                1.02,
                0.5,
            ),
            title="Component",
            title_fontsize=16,
            alignment="left",
            fontsize=16,
            frameon=False,
        )

        plt.tight_layout()
        plt.show()

        return "Class change components chart rendered successfully."

### 4.6 Maps

In [ ]:
# @title 4.6.1 Maps Processing Module
# @markdown Plots the Number of Changes, Trajectories, Extent Change, Alternation Exchange and Alternation Shift maps from the rasters written by the QUEST core engine, read at a display-safe resolution, with a scale bar measured on the ground.

# Quest Maps Processing Engine
class QuestMaps(QuestChangeComponents):
    """
    Handles the creation and visualization of spatial cartographic maps,
    including pixel-level change frequency tracking and trajectories over time.
    """

    def _read_map_raster(
        self,
        raster_path: str,
        max_display_dim: int = 2000,
    ) -> tuple:
        """
        Read a raster at a display-safe resolution for plotting.

        Parameters
        ----------
        raster_path : str
            Path to the raster map.
        max_display_dim : int, optional
            Maximum size, in pixels, of the longer dimension read into
            memory. Larger rasters are decimated to this size.

        Returns
        -------
        tuple
            Masked array, transform of the decimated grid, CRS, and the
            ground size in kilometers of one displayed pixel.
        """
        with rasterio.open(
            raster_path,
        ) as src:
            downsample = max(
                1,
                max(src.height, src.width) // max_display_dim,
            )
            out_height = max(
                1,
                src.height // downsample,
            )
            out_width = max(
                1,
                src.width // downsample,
            )
            data = src.read(
                1,
                out_shape=(
                    out_height,
                    out_width,
                ),
                resampling=rasterio.enums.Resampling.nearest,
            )
            data = np.ma.masked_equal(
                data,
                self.no_data_value,
            )
            src_crs = src.crs
            # Rebuild the transform for the decimated grid, so pixel
            # coordinates still map to the correct location
            transform = rasterio.transform.from_bounds(
                *src.bounds,
                out_width,
                out_height,
            )

        # Ground size of one displayed pixel, which covers `downsample`
        # native pixels in each direction
        pixel_size_km = compute_display_pixel_size_km(
            raster_path=raster_path,
            downsample_divisor=downsample,
        )

        return data, transform, src_crs, pixel_size_km

    def compute_number_of_changes_raster(
        self,
    ) -> str:
        """
        Return the number of changes raster written by the QUEST core engine.

        Returns
        -------
        str
            The file path to the raster map.
        """
        cached = self.get_quest_results()
        return cached["results"]["out_raster_paths"]["number_of_changes"]

    def plot_number_of_changes_map(
        self,
    ) -> str:
        """
        Plots the number of changes spatial distribution and exports it as a PNG image.

        Returns
        -------
        str
            A confirmation message denoting the plot completion and saved file location.
        """
        raster_to_plot_path = self.compute_number_of_changes_raster()
        data, transform, src_crs, pixel_size_km = self._read_map_raster(
            raster_path=raster_to_plot_path,
        )

        fig, ax = plt.subplots(
            figsize=(
                6,
                8,
            ),
            dpi=300,
        )
        min_val = int(
            np.ma.min(
                data,
            )
        )
        max_val = int(
            np.ma.max(
                data,
            )
        )

        original_cmap = plt.get_cmap(
            "viridis_r",
        )
        color_list = [
            "#c0c0c0"
        ] + [
            original_cmap(i)
            for i in np.linspace(
                0,
                1,
                max_val,
            )
        ]
        cmap = mcolors.ListedColormap(
            color_list,
        )
        bounds = np.arange(
            min_val,
            max_val + 2,
        ) - 0.5
        norm = mcolors.BoundaryNorm(
            bounds,
            cmap.N,
        )

        ax.imshow(
            data,
            cmap=cmap,
            interpolation="nearest",
            norm=norm,
        )

        legend_elements = []
        present_values = np.unique(
            data.compressed(),
        )
        for i in range(
            min_val,
            max_val + 1,
        ):
            if i in present_values:
                legend_elements.append(
                    Patch(
                        facecolor=cmap(
                            norm(
                                i,
                            )
                        ),
                        edgecolor="none",
                        linewidth=0,
                        label=str(
                            i,
                        ),
                    )
                )

        ax.legend(
            handles=legend_elements,
            title="Changes",
            loc="center left",
            bbox_to_anchor=(
                1.02,
                0.5,
            ),
            frameon=False,
            fontsize=8,
            title_fontsize=10,
            alignment="left",
            handlelength=2.0,
            handleheight=1.5,
        )

        scalebar = ScaleBar(
            dx=pixel_size_km,
            units="km",
            length_fraction=0.35,
            location="lower right",
            box_alpha=0.0,
            scale_formatter=format_scale_bar_label,
        )
        ax.add_artist(
            scalebar,
        )
        north_arrow(
            ax,
            location="upper left",
            shadow=False,
            rotation={
                "degrees": 0,
            },
            scale=0.3,
        )

        ax.set_title(
            "Number of Changes",
            fontsize=18,
            pad=5,
        )
        ax.set_aspect(
            "equal",
        )
        to_latlon = Transformer.from_crs(
            src_crs,
            "EPSG:4326",
            always_xy=True,
        )
        height, width = data.shape

        def format_lon(x, pos):
            x = np.clip(
                x,
                0,
                width - 1,
            )
            x_proj, y_proj = rasterio.transform.xy(
                transform,
                height // 2,
                x,
            )
            lon, lat = to_latlon.transform(
                x_proj,
                y_proj,
            )
            return f"{lon:.1f}°"

        def format_lat(y, pos):
            y = np.clip(
                y,
                0,
                height - 1,
            )
            x_proj, y_proj = rasterio.transform.xy(
                transform,
                y,
                width // 2,
            )
            lon, lat = to_latlon.transform(
                x_proj,
                y_proj,
            )
            return f"{lat:.1f}°"

        ax.xaxis.set_major_formatter(
            FuncFormatter(
                format_lon,
            )
        )
        ax.yaxis.set_major_formatter(
            FuncFormatter(
                format_lat,
            )
        )
        ax.xaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=4,
            )
        )
        ax.yaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=6,
            )
        )
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=7,
            pad=4,
        )
        plt.setp(
            ax.get_yticklabels(),
            rotation=90,
            va="center",
        )

        maps_dir = os.path.join(
            self.get_quest_results()["run_dir"],
            "maps",
        )
        os.makedirs(
            maps_dir,
            exist_ok=True,
        )
        output_figure_path = os.path.join(
            maps_dir,
            "map_number_of_changes.png",
        )

        plt.savefig(
            output_figure_path,
            dpi=300,
            bbox_inches="tight",
            format="png",
            pad_inches=0.5,
        )
        plt.show()

        return f"Number of changes map rendered and saved to {output_figure_path}"

    # Raster key in the engine results, map title and legend title of each
    # component map, as in the 03_quest_pie notebook
    COMPONENT_MAPS = {
        "extent": ("Extent Change", "Change"),
        "alternation_exchange": ("Alternation Exchange", "Exchange"),
        "alternation_shift": ("Alternation Shift", "Shift"),
    }
    COMPONENT_RASTER_KEYS = {
        "extent": "extent",
        "alternation_exchange": "exchange",
        "alternation_shift": "shift",
    }

    def compute_component_raster(
        self,
        component: str,
    ) -> str:
        """
        Return the raster of one change component written by the QUEST core
        engine.

        Parameters
        ----------
        component : str
            "extent", "alternation_exchange" or "alternation_shift".

        Returns
        -------
        str
            The file path to the raster map.
        """
        cached = self.get_quest_results()
        key = self.COMPONENT_RASTER_KEYS[component]
        return cached["results"]["out_raster_paths"][key]

    def count_raster_values(
        self,
        raster_path: str,
    ) -> dict:
        """
        Count the locations of each value of a raster, block by block, so
        large rasters are never read whole. NoData is left out.

        Parameters
        ----------
        raster_path : str
            Path to the raster map.

        Returns
        -------
        dict
            Mapping from raster value to number of locations.
        """
        counts = np.zeros(
            256,
            dtype=np.int64,
        )
        with rasterio.open(
            raster_path,
        ) as src:
            for _, window in src.block_windows(1):
                block = src.read(
                    1,
                    window=window,
                )
                counts += np.bincount(
                    block.ravel(),
                    minlength=256,
                )[:256]

        counts[self.no_data_value] = 0
        return {
            int(value): int(count)
            for value, count in enumerate(counts)
            if count > 0
        }

    def plot_component_map(
        self,
        component: str,
    ) -> str:
        """
        Plots the map of one change component and exports it as a PNG image.

        Parameters
        ----------
        component : str
            "extent", where 1 marks a location whose class differs between
            the first and last time points; "alternation_exchange" or
            "alternation_shift", the number of changes of each location in
            that component.

        Returns
        -------
        str
            A confirmation message denoting the plot completion and saved file location.
        """
        title, legend_title = self.COMPONENT_MAPS[component]
        raster_to_plot_path = self.compute_component_raster(
            component,
        )
        data, transform, src_crs, pixel_size_km = self._read_map_raster(
            raster_path=raster_to_plot_path,
        )

        fig, ax = plt.subplots(
            figsize=(
                6,
                8,
            ),
            dpi=300,
        )
        min_val = int(
            np.ma.min(
                data,
            )
        )
        max_val = int(
            np.ma.max(
                data,
            )
        )

        original_cmap = plt.get_cmap(
            "viridis_r",
        )
        color_list = [
            "#c0c0c0"
        ] + [
            original_cmap(i)
            for i in np.linspace(
                0,
                1,
                max_val,
            )
        ]
        cmap = mcolors.ListedColormap(
            color_list,
        )
        bounds = np.arange(
            min_val,
            max_val + 2,
        ) - 0.5
        norm = mcolors.BoundaryNorm(
            bounds,
            cmap.N,
        )

        ax.imshow(
            data,
            cmap=cmap,
            interpolation="nearest",
            norm=norm,
        )

        legend_elements = []
        present_values = np.unique(
            data.compressed(),
        )
        for i in range(
            min_val,
            max_val + 1,
        ):
            if i in present_values:
                legend_elements.append(
                    Patch(
                        facecolor=cmap(
                            norm(
                                i,
                            )
                        ),
                        edgecolor="none",
                        linewidth=0,
                        label=str(
                            i,
                        ),
                    )
                )

        ax.legend(
            handles=legend_elements,
            title=legend_title,
            loc="center left",
            bbox_to_anchor=(
                1.02,
                0.5,
            ),
            frameon=False,
            fontsize=8,
            title_fontsize=10,
            alignment="left",
            handlelength=2.0,
            handleheight=1.5,
        )

        scalebar = ScaleBar(
            dx=pixel_size_km,
            units="km",
            length_fraction=0.35,
            location="lower right",
            box_alpha=0.0,
            scale_formatter=format_scale_bar_label,
        )
        ax.add_artist(
            scalebar,
        )
        north_arrow(
            ax,
            location="upper left",
            shadow=False,
            rotation={
                "degrees": 0,
            },
            scale=0.3,
        )

        ax.set_title(
            title,
            fontsize=18,
            pad=5,
        )
        ax.set_aspect(
            "equal",
        )
        to_latlon = Transformer.from_crs(
            src_crs,
            "EPSG:4326",
            always_xy=True,
        )
        height, width = data.shape

        def format_lon(x, pos):
            x = np.clip(
                x,
                0,
                width - 1,
            )
            x_proj, y_proj = rasterio.transform.xy(
                transform,
                height // 2,
                x,
            )
            lon, lat = to_latlon.transform(
                x_proj,
                y_proj,
            )
            return f"{lon:.1f}°"

        def format_lat(y, pos):
            y = np.clip(
                y,
                0,
                height - 1,
            )
            x_proj, y_proj = rasterio.transform.xy(
                transform,
                y,
                width // 2,
            )
            lon, lat = to_latlon.transform(
                x_proj,
                y_proj,
            )
            return f"{lat:.1f}°"

        ax.xaxis.set_major_formatter(
            FuncFormatter(
                format_lon,
            )
        )
        ax.yaxis.set_major_formatter(
            FuncFormatter(
                format_lat,
            )
        )
        ax.xaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=4,
            )
        )
        ax.yaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=6,
            )
        )
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=7,
            pad=4,
        )
        plt.setp(
            ax.get_yticklabels(),
            rotation=90,
            va="center",
        )

        maps_dir = os.path.join(
            self.get_quest_results()["run_dir"],
            "maps",
        )
        os.makedirs(
            maps_dir,
            exist_ok=True,
        )
        output_figure_path = os.path.join(
            maps_dir,
            f"map_{component}.png",
        )

        plt.savefig(
            output_figure_path,
            dpi=300,
            bbox_inches="tight",
            format="png",
            pad_inches=0.5,
        )
        plt.show()

        return f"{title} map rendered and saved to {output_figure_path}"

    def compute_trajectory_raster(
        self,
    ) -> str:
        """
        Return the trajectory raster written by the QUEST core engine.

        Returns
        -------
        str
            The file path to the trajectory raster map.
        """
        cached = self.get_quest_results()
        return cached["results"]["out_raster_paths"]["trajectory"]

    def plot_trajectory_map(
        self,
    ) -> str:
        """
        Plot the compiled Trajectory raster map with cartographic visualization elements.

        Returns
        -------
        str
            A confirmation message indicating successful plot and location of the saved image.
        """
        raster_to_plot_path = self.compute_trajectory_raster()
        data, transform, src_crs, pixel_size_km = self._read_map_raster(
            raster_path=raster_to_plot_path,
        )

        fig, ax = plt.subplots(
            figsize=(
                6,
                8,
            ),
            dpi=300,
        )

        colors_dict = {
            1: "#d9d9d9",
            2: "#990033",
            3: "#FDE725",
            4: "#ff9900",
            5: "#000066",
        }

        labels_dict = {
            1: "1: Changes = 0",
            2: "2: Changes > 1 with stable extent",
            3: "3: Changes = 1",
            4: "4: Changes > 2 with direct transition",
            5: "5: Changes > 1 without direct transition",
        }

        cmap = mcolors.ListedColormap(
            [
                colors_dict.get(
                    i,
                    "#ffffff",
                )
                for i in range(
                    1,
                    6,
                )
            ]
        )
        bounds = np.arange(
            0.5,
            6.5,
            1,
        )
        norm = mcolors.BoundaryNorm(
            bounds,
            cmap.N,
        )

        ax.imshow(
            data,
            cmap=cmap,
            interpolation="nearest",
            norm=norm,
        )

        legend_elements = []
        present_values = np.unique(
            data.compressed(),
        )

        for val in sorted(
            present_values,
        ):
            if val in colors_dict:
                legend_elements.append(
                    Patch(
                        facecolor=colors_dict[
                            val
                        ],
                        edgecolor="black",
                        linewidth=0,
                        label=labels_dict[
                            val
                        ],
                    )
                )

        ax.legend(
            handles=legend_elements,
            title="Trajectories",
            loc="center left",
            bbox_to_anchor=(
                1.02,
                0.5,
            ),
            frameon=False,
            fontsize=8,
            title_fontsize=10,
            alignment="left",
            handlelength=2.0,
            handleheight=1.5,
        )

        scalebar = ScaleBar(
            dx=pixel_size_km,
            units="km",
            length_fraction=0.35,
            location="lower right",
            box_alpha=0.0,
            scale_formatter=format_scale_bar_label,
        )
        ax.add_artist(
            scalebar,
        )
        north_arrow(
            ax,
            location="upper left",
            shadow=False,
            rotation={
                "degrees": 0,
            },
            scale=0.3,
        )

        ax.set_title(
            "Trajectories Map",
            fontsize=18,
            pad=5,
        )
        ax.set_aspect(
            "equal",
        )
        to_latlon = Transformer.from_crs(
            src_crs,
            "EPSG:4326",
            always_xy=True,
        )
        height, width = data.shape

        def format_lon(x, pos):
            x = np.clip(
                x,
                0,
                width - 1,
            )
            x_proj, y_proj = rasterio.transform.xy(
                transform,
                height // 2,
                x,
            )
            lon, lat = to_latlon.transform(
                x_proj,
                y_proj,
            )
            return f"{lon:.1f}°"

        def format_lat(y, pos):
            y = np.clip(
                y,
                0,
                height - 1,
            )
            x_proj, y_proj = rasterio.transform.xy(
                transform,
                y,
                width // 2,
            )
            lon, lat = to_latlon.transform(
                x_proj,
                y_proj,
            )
            return f"{lat:.1f}°"

        ax.xaxis.set_major_formatter(
            FuncFormatter(
                format_lon,
            )
        )
        ax.yaxis.set_major_formatter(
            FuncFormatter(
                format_lat,
            )
        )
        ax.xaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=4,
            )
        )
        ax.yaxis.set_major_locator(
            mticker.MaxNLocator(
                nbins=6,
            )
        )
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=7,
            pad=4,
        )
        plt.setp(
            ax.get_yticklabels(),
            rotation=90,
            va="center",
        )

        maps_dir = os.path.join(
            self.get_quest_results()["run_dir"],
            "maps",
        )
        os.makedirs(
            maps_dir,
            exist_ok=True,
        )
        output_figure_path = os.path.join(
            maps_dir,
            "map_trajectories.png",
        )

        plt.savefig(
            output_figure_path,
            dpi=300,
            bbox_inches="tight",
            format="png",
            pad_inches=0.5,
        )
        plt.show()

        return f"Trajectory map rendered and saved to {output_figure_path}"

## 5. QUEST chat

In [ ]:
# @title 5.1 QuestEngine Chat Interface
# @markdown Initializes the master agentic AI loop with access to ALL tools and explicit docstrings from all modules (Class Size, Number of Changes, Trajectories, Transition Matrices, Change Components, Maps). Every question, answer and tool call is written to a log file in the output folder.

# 1. Check the key and initialize all analyzers with the settings from Section 2
if "GEMINI_API_KEY" not in os.environ:
    raise RuntimeError("GEMINI_API_KEY not available. Run cell 1.3 first.")

analyzer_class = QuestClassSize(QUEST_CONFIG)
analyzer_changes = QuestNumberChanges(QUEST_CONFIG)
analyzer_traj = QuestTrajectory(QUEST_CONFIG)
analyzer_trans = QuestTransition(QUEST_CONFIG)
analyzer_comp = QuestChangeComponents(QUEST_CONFIG)
analyzer_maps = QuestMaps(QUEST_CONFIG)

# 2. Record every tool call made while the agent answers a question. The
#    record is written by the code, not by the model, so the model cannot
#    leave out or invent a call.
_tool_calls = []


@contextlib.contextmanager
def save_shown_figures(
    tool_name: str,
):
    """
    While a chart tool runs, save each figure to OUTPUT_PATH/charts before
    it is shown, and collect the paths of the files written.
    """
    charts_dir = os.path.join(
        OUTPUT_PATH,
        "charts",
    )
    os.makedirs(
        charts_dir,
        exist_ok=True,
    )
    saved = []
    original_show = plt.show

    def show_and_save(
        *args,
        **kwargs,
    ):
        path = os.path.join(
            charts_dir,
            f"{datetime.now():%Y%m%d_%H%M%S}_{tool_name}.png",
        )
        plt.savefig(
            path,
            dpi=300,
            bbox_inches="tight",
        )
        saved.append(
            path,
        )
        original_show(
            *args,
            **kwargs,
        )

    plt.show = show_and_save
    try:
        yield saved
    finally:
        plt.show = original_show


def log_tool_call(
    func,
):
    """
    Wrap a tool so each call is recorded with its arguments, run time and
    status. A tool that raises an exception returns a short error instead,
    and a result with an "error" key is recorded as a failure. Charts drawn
    by a chart tool are saved to OUTPUT_PATH/charts, and the result gives
    the file in "chart_file"; map tools save their own files.

    Parameters
    ----------
    func : callable
        Tool given to the agent.

    Returns
    -------
    callable
        The same tool, with the same name, signature and docstring, so the
        schema seen by the model does not change.
    """

    @functools.wraps(
        func,
    )
    def wrapper(
        *args,
        **kwargs,
    ):
        record = {
            "tool": func.__name__,
            "arguments": list(args) if args else dict(kwargs),
        }
        start = time.perf_counter()
        saves_chart = (
            func.__name__.startswith("plot_")
            and not func.__name__.endswith("_map")
        )
        try:
            if saves_chart:
                with save_shown_figures(func.__name__) as saved:
                    result = func(
                        *args,
                        **kwargs,
                    )
                if saved and isinstance(result, dict) and "error" not in result:
                    result["chart_file"] = saved[-1]
            else:
                result = func(
                    *args,
                    **kwargs,
                )
        except Exception as error:
            # The agent receives a short message it can explain, instead of
            # the long text the client library builds from an exception
            record["status"] = "error"
            record["error"] = f"{type(error).__name__}: {error}"
            return {
                "error": (
                    f"The tool {func.__name__} failed with "
                    f"{record['error']}"
                ),
            }
        else:
            # A tool can also report a problem by returning an error
            if isinstance(result, dict) and "error" in result:
                record["status"] = "error"
                record["error"] = result["error"]
            else:
                record["status"] = "ok"
            return result
        finally:
            record["seconds"] = round(
                time.perf_counter() - start,
                3,
            )
            _tool_calls.append(
                record,
            )

    return wrapper


# 3. Convert the results of each module to one format returned to the agent.
#    Every data tool and every chart tool returns a dictionary with the unit,
#    the time extent, the study area, readable column names and the rows,
#    so the agent can read the numbers without a second call.
TRAJECTORY_LABELS = {
    1: "Changes = 0",
    2: "Changes > 1 with stable extent",
    3: "Changes = 1",
    4: "Changes > 2 with direct transition",
    5: "Changes > 1 without direct transition",
}


def compute_pixel_area(
    raster_path: str,
) -> dict:
    """
    Compute the area of one pixel of the input rasters.

    In a projected coordinate reference system the area is the spatial
    resolution along x times the spatial resolution along y, converted to
    square meters, and it is exact. In a geographic coordinate reference
    system the area changes with latitude, so the geodesic area of the pixel
    at the center of the map is used, and the result says it is approximate
    when the area changes by more than 1% between the top and bottom rows.

    Parameters
    ----------
    raster_path : str
        Path to one of the input rasters.

    Returns
    -------
    dict
        "m2" : float, area of one pixel in square meters.
        "approximate" : bool, True when the area changes across the map by
            more than 1%.
        "note" : str, how the area was computed.
    """
    with rasterio.open(
        raster_path,
    ) as src:
        crs = CRS.from_user_input(
            src.crs,
        )
        res_x, res_y = src.res
        left, bottom, right, top = src.bounds

    if not crs.is_geographic:
        to_meters = crs.axis_info[0].unit_conversion_factor
        return {
            "m2": res_x * res_y * to_meters ** 2,
            "approximate": False,
            "note": "Exact area from the spatial resolution of the rasters.",
        }

    geod = Geod(
        ellps="WGS84",
    )
    x_center = (left + right) / 2.0

    def _pixel_area_m2(
        y_center: float,
    ) -> float:
        lons = [
            x_center - res_x / 2.0,
            x_center + res_x / 2.0,
            x_center + res_x / 2.0,
            x_center - res_x / 2.0,
        ]
        lats = [
            y_center - res_y / 2.0,
            y_center - res_y / 2.0,
            y_center + res_y / 2.0,
            y_center + res_y / 2.0,
        ]
        area, _ = geod.polygon_area_perimeter(
            lons,
            lats,
        )
        return abs(area)

    area_center = _pixel_area_m2(
        (top + bottom) / 2.0,
    )
    areas = [
        area_center,
        _pixel_area_m2(top - res_y / 2.0),
        _pixel_area_m2(bottom + res_y / 2.0),
    ]
    variation_pct = (max(areas) - min(areas)) / area_center * 100.0
    approximate = variation_pct > 1.0

    note = "Geodesic area of one pixel at the center of the map."
    if approximate:
        note += (
            f" The pixel area changes by {variation_pct:.1f}% between the top "
            f"and bottom rows, so areas are approximate."
        )

    return {
        "m2": area_center,
        "approximate": approximate,
        "note": note,
    }


PIXEL_AREA = compute_pixel_area(
    next(iter(QUEST_CONFIG["raster_by_year"].values())),
)


def add_area_columns(
    row: dict,
) -> dict:
    """
    Add the area in square kilometers and hectares after every count of
    locations in a row. A column "pixels" gets "area_km2" and "area_ha"; a
    column "gain_pixels" gets "gain_km2" and "gain_ha".
    """
    out = {}
    for key, value in row.items():
        out[key] = value
        if key == "pixels" or key.endswith("_pixels"):
            prefix = "area" if key == "pixels" else key[: -len("_pixels")]
            area_m2 = value * PIXEL_AREA["m2"]
            out[f"{prefix}_km2"] = round(area_m2 / 1e6, 4)
            out[f"{prefix}_ha"] = round(area_m2 / 1e4, 2)

    return out


def _percent(
    pixels: float,
    study_area_pixels: int,
) -> float:
    """Share of the study area, in percent, rounded to four decimals."""
    if study_area_pixels <= 0:
        return 0.0

    return round(
        float(pixels) / study_area_pixels * 100.0,
        4,
    )


def build_tool_result(
    analyzer,
    rows: list[dict],
    chart_description: str | None = None,
    **extra,
) -> dict:
    """
    Wrap the rows of a tool in the format returned to the agent.

    Parameters
    ----------
    analyzer : QuestBase
        Module that produced the rows. Gives the time extent and the study
        area.
    rows : list of dict
        Records with readable column names.
    chart_description : str, optional
        Text returned by a chart method. When given, the result says that a
        chart was shown and carries this text.
    **extra
        Other fields to add to the result, such as a note on the columns.

    Returns
    -------
    dict
        "unit", "time_extent", "time_points", "study_area_pixels", the
        extra fields, "columns", "rows" with one list of values per row
        and, for charts, "chart_shown" and "chart_description".
    """
    years = analyzer.years
    study_area = analyzer.get_quest_results()["study_area_pixels"]
    result = {
        "unit": "pixels",
        "time_extent": f"{years[0]}-{years[-1]}",
        "time_points": len(years),
        "study_area_pixels": study_area,
        "study_area_km2": round(study_area * PIXEL_AREA["m2"] / 1e6, 4),
        "study_area_ha": round(study_area * PIXEL_AREA["m2"] / 1e4, 2),
    }
    if PIXEL_AREA["approximate"]:
        result["area_note"] = PIXEL_AREA["note"]
    result.update(
        extra,
    )
    rows = [
        add_area_columns(row)
        for row in rows
    ]

    # Rows are sent as a header and lists of values, which takes about a
    # quarter of the tokens of one dictionary per row
    result["columns"] = list(rows[0]) if rows else []
    result["rows"] = [
        list(row.values())
        for row in rows
    ]

    if chart_description is not None:
        result["chart_shown"] = True
        result["chart_description"] = chart_description

    return result


def _intervals_within(
    intervals,
    start_year: int | None,
    end_year: int | None,
) -> list[str]:
    """
    Keep the intervals inside start_year and end_year, with the same rule as
    the interval charts. Without both years, keep every interval.
    """
    if start_year is None or end_year is None:
        return list(
            intervals,
        )

    kept = []
    for interval in intervals:
        y_start, y_end = (
            int(year)
            for year in str(interval).split("-")
        )
        if y_start >= start_year and y_end <= end_year:
            kept.append(
                interval,
            )

    return kept


def class_size_rows(
    analyzer,
    specific_years: list[int] | None = None,
) -> list[dict]:
    """Pixels of each class at each year, optionally for some years only."""
    study_area = analyzer.get_quest_results()["study_area_pixels"]
    rows = []

    for year, counts in analyzer.compute_class_pixels_over_time().items():
        if specific_years and year not in specific_years:
            continue
        for class_name, pixels in counts.items():
            rows.append(
                {
                    "year": int(year),
                    "class": class_name,
                    "pixels": int(pixels),
                    "percent": _percent(pixels, study_area),
                },
            )

    return rows


def number_of_changes_interval_rows(
    analyzer,
    start_year: int | None = None,
    end_year: int | None = None,
) -> list[dict]:
    """Changing pixels in each interval, by number of changes of the pixel."""
    df = analyzer.compute_number_of_changes()
    rows = []

    for interval in _intervals_within(df.index, start_year, end_year):
        for n_changes in df.columns:
            rows.append(
                {
                    "interval": str(interval),
                    "number_of_changes": int(n_changes),
                    "pixels": int(df.loc[interval, n_changes]),
                },
            )

    return rows


def overall_changes_rows(
    analyzer,
) -> list[dict]:
    """Pixels by the number of changes over the whole time extent."""
    df = analyzer.compute_overall_changes()

    return [
        {
            "number_of_changes": int(n_changes),
            "pixels": int(row["absolute_pixels"]),
            "percent": round(float(row["percentage"]), 4),
        }
        for n_changes, row in df.iterrows()
    ]


def trajectory_interval_rows(
    analyzer,
    start_year: int | None = None,
    end_year: int | None = None,
) -> list[dict]:
    """Changing pixels in each interval, by trajectory of the pixel."""
    df = analyzer.compute_trajectory_intervals()
    rows = []

    for interval in _intervals_within(df.index, start_year, end_year):
        for trajectory in df.columns:
            rows.append(
                {
                    "interval": str(interval),
                    "trajectory": int(trajectory),
                    "trajectory_label": TRAJECTORY_LABELS[int(trajectory)],
                    "pixels": int(df.loc[interval, trajectory]),
                },
            )

    return rows


def overall_trajectory_rows(
    analyzer,
    for_chart: bool = False,
) -> list[dict]:
    """
    Pixels of each trajectory over the whole time extent, trajectories 1 to
    5. For a chart, a column says which trajectories the chart shows, since
    the trajectory charts leave out trajectory 1.
    """
    cached = analyzer.get_quest_results()
    trajectory_counts = cached["results"]["trajectory_counts"]
    study_area = cached["study_area_pixels"]
    rows = []

    for trajectory in range(1, 6):
        row = {
            "trajectory": trajectory,
            "trajectory_label": TRAJECTORY_LABELS[trajectory],
            "pixels": int(trajectory_counts[trajectory]),
            "percent": _percent(trajectory_counts[trajectory], study_area),
        }
        if for_chart:
            row["in_chart"] = trajectory != 1
        rows.append(
            row,
        )

    return rows


def matrix_rows(
    analyzer,
    df: pd.DataFrame,
) -> list[dict]:
    """Transition matrix as from-class, to-class and pixels."""
    names = {
        code: info["name"]
        for code, info in analyzer.class_labels_dict.items()
    }

    return [
        {
            "from_class": names.get(from_code, str(from_code)),
            "to_class": names.get(to_code, str(to_code)),
            "pixels": int(df.loc[from_code, to_code]),
        }
        for from_code in df.index
        for to_code in df.columns
    ]


def component_rows(
    analyzer,
    df: pd.DataFrame,
) -> list[dict]:
    """Rows of the change components table with readable names."""
    study_area = analyzer.get_quest_results()["study_area_pixels"]

    return [
        {
            "time_interval": row["Time_Interval"],
            "class": row["Class"],
            "component": row["Component"],
            "gain_pixels": int(row["Gain"]),
            "loss_pixels": int(row["Loss"]),
            "gain_percent": _percent(row["Gain"], study_area),
            "loss_percent": _percent(row["Loss"], study_area),
        }
        for _, row in df.iterrows()
    ]


def component_interval_rows(
    analyzer,
) -> list[dict]:
    """Size of each component in each interval, summed over the classes."""
    df = analyzer.compute_change_components_table()
    study_area = analyzer.get_quest_results()["study_area_pixels"]
    time_df = df[
        df["Time_Interval"].str.contains(
            "-",
            na=False,
        )
    ]
    totals = time_df.groupby(
        [
            "Time_Interval",
            "Component",
        ],
    )["Gain"].sum()

    return [
        {
            "interval": interval,
            "component": component,
            "pixels": int(pixels),
            "percent": _percent(pixels, study_area),
        }
        for (interval, component), pixels in totals.items()
    ]


def component_overall_rows(
    analyzer,
) -> list[dict]:
    """Size of each component over the whole time extent."""
    df = summarize_change_components(
        analyzer.compute_change_components_table(),
    )
    study_area = analyzer.get_quest_results()["study_area_pixels"]

    return [
        {
            "component": row["Component"],
            "pixels": int(row["Pixels"]),
            "percent": _percent(row["Pixels"], study_area),
        }
        for _, row in df.iterrows()
    ]


def component_class_rows(
    analyzer,
) -> list[dict]:
    """Gain and loss of each class in each component, for the whole extent."""
    df = analyzer.compute_change_components_table()
    df = df[
        df["Time_Interval"].isin(
            [
                "extent",
                "alternation_exchange",
                "alternation_shift",
            ],
        )
    ]

    return component_rows(
        analyzer,
        df,
    )


NUMBER_OF_CHANGES_NOTE = (
    "number_of_changes is the total number of changes of the pixel over the "
    "whole time extent."
)


def check_metric(
    metric: str,
) -> dict | None:
    """Return an error when the chart metric is not supported."""
    if metric in (
        "percentage",
        "pixels",
    ):
        return None

    return {
        "error": (
            f"The metric '{metric}' is not supported. Use 'percentage' or "
            f"'pixels'. Charts do not plot areas, but every result gives the "
            f"area of each row in area_km2 and area_ha."
        ),
    }


def check_year_range(
    analyzer,
    start_year: int | None,
    end_year: int | None,
) -> dict | None:
    """
    Return an error when a time extent cannot be analyzed: only one of the
    two years given, years in the wrong order, or fewer than two years of
    the dataset inside the extent.
    """
    if start_year is None and end_year is None:
        return None

    years = analyzer.years
    available = f"The dataset has the years {years[0]} to {years[-1]}."

    if start_year is None or end_year is None:
        return {
            "error": (
                f"Give both start_year and end_year, or neither. {available}"
            ),
        }

    if start_year >= end_year:
        return {
            "error": (
                f"start_year {start_year} must come before end_year "
                f"{end_year}. {available}"
            ),
        }

    years_inside = [
        year
        for year in years
        if start_year <= year <= end_year
    ]
    if len(years_inside) < 2:
        return {
            "error": (
                f"Fewer than two years of the dataset fall between "
                f"{start_year} and {end_year}. {available}"
            ),
        }

    return None


def check_specific_years(
    analyzer,
    specific_years: list[int] | None,
) -> dict | None:
    """Return an error when a requested year is not in the dataset."""
    if not specific_years:
        return None

    years = analyzer.years
    missing = [
        year
        for year in specific_years
        if year not in years
    ]
    if not missing:
        return None

    return {
        "error": (
            f"The years {missing} are not in the dataset, which has the "
            f"years {years[0]} to {years[-1]}."
        ),
    }


# 4. Unified Tool Definitions with Explicit Docstrings

# --- Dataset Metadata Tool ---
AGGREGATED_MATRIX_KEYS = [
    "time_intervals",
    "extent",
    "allocation_exchange",
    "quantity_allocation_shift",
    "alternation_exchange",
    "alternation_shift",
    "indirect_extent",
]


def get_dataset_metadata() -> dict:
    """
    Retrieves the description of the dataset: years, time intervals, land
    cover classes, NoData value, valid transition matrix keys, trajectory
    labels, raster grid, area of one pixel and the warnings from the input
    check. Call this
    tool before using a year, an interval, a class name or a matrix key that
    you have not seen in a tool result. It does not run the analysis.
    """
    years = analyzer_class.years
    intervals = [
        f"{start}-{end}"
        for start, end in zip(years[:-1], years[1:])
    ]
    report = QUEST_CONFIG.get("input_report", {})

    return {
        "years": years,
        "time_extent": f"{years[0]}-{years[-1]}",
        "time_points": len(years),
        "intervals": intervals,
        "max_number_of_changes": len(intervals),
        "classes": [
            {"code": int(code), "name": info["name"]}
            for code, info in analyzer_class.class_labels_dict.items()
        ],
        "no_data_value": QUEST_CONFIG["no_data_value"],
        "matrix_keys": (
            AGGREGATED_MATRIX_KEYS
            + [f"trajectory_{trajectory}" for trajectory in range(1, 6)]
            + [f"interval_{interval}" for interval in intervals]
        ),
        "trajectory_labels": {
            str(trajectory): label
            for trajectory, label in TRAJECTORY_LABELS.items()
        },
        "grid": {
            "width_pixels": report.get("width"),
            "height_pixels": report.get("height"),
            "crs": report.get("crs"),
            "spatial_resolution": report.get("spatial_resolution"),
        },
        "pixel_area": {
            "m2": round(PIXEL_AREA["m2"], 4),
            "approximate": PIXEL_AREA["approximate"],
            "note": PIXEL_AREA["note"],
        },
        "input_warnings": report.get("warnings", []),
    }

# --- Class Size Tools ---
def get_class_pixels_over_time() -> dict:
    """
    Returns the number of locations of each class at every time point, with
    each class as a percent of the study area. Use it for questions about
    class sizes, gains or losses between years, when no chart is asked for.
    """
    return build_tool_result(
        analyzer_class,
        class_size_rows(analyzer_class),
    )

def get_class_summary(class_name: str, start_year: int = None, end_year: int = None) -> dict:
    """
    Returns everything about one class in a single result: its size at the
    first and last time points and the net change; its gain and loss summed
    over every time interval; its gain and loss in each component of the
    whole time extent; and its transitions from and to the other classes
    between the first and last time points. Use it for questions about one
    class instead of calling several tools.

    Parameters
    ----------
    class_name : str
        Name of the class, as listed by get_dataset_metadata.
    start_year, end_year : int, optional
        Run the analysis again for the years between them, a custom time
        extent with its own study area. Give both or neither; without them,
        the result covers the whole time extent.
    """
    names = {
        info["name"].lower(): (code, info["name"])
        for code, info in analyzer_class.class_labels_dict.items()
    }
    if class_name.strip().lower() not in names:
        return {
            "error": (
                f"The class '{class_name}' is not in the dataset. The classes "
                f"are {[name for _, name in names.values()]}."
            ),
        }
    code, name = names[class_name.strip().lower()]

    error = check_year_range(analyzer_comp, start_year, end_year)
    if error:
        return error
    analyzer = period_analyzer(analyzer_comp, start_year, end_year)

    cached = analyzer.get_quest_results()
    study_area = cached["study_area_pixels"]
    years = analyzer.years
    class_counts = dict(
        zip(
            cached["results"]["class_ids"],
            zip(
                cached["results"]["class_counts"][0],
                cached["results"]["class_counts"][-1],
            ),
        ),
    )

    # 1. Size at the first and last time points
    first, last = (
        int(count)
        for count in class_counts.get(code, (0, 0))
    )
    size_rows = [
        {
            "time_point": label,
            "year": year,
            "pixels": int(pixels),
            "percent": _percent(pixels, study_area),
        }
        for label, year, pixels in [
            ("first", years[0], first),
            ("last", years[-1], last),
        ]
    ]

    # 2. Net change, and gain and loss summed over every time interval
    intervals = cached["tables"]["aggregated_matrices"]["time_intervals"]
    change_table = [
        {
            "measure": "net change from the first to the last time point",
            "pixels": int(last - first),
            "percent": _percent(last - first, study_area),
        },
        {
            "measure": "gain summed over every time interval",
            "pixels": int(intervals[code].sum() - intervals.loc[code, code]),
        },
        {
            "measure": "loss summed over every time interval",
            "pixels": int(intervals.loc[code].sum() - intervals.loc[code, code]),
        },
    ]
    for row in change_table[1:]:
        row["percent"] = _percent(row["pixels"], study_area)

    # 3. Gain and loss in each component of the whole time extent
    components = analyzer.compute_change_components_table()
    component_names = {
        ("extent", "Allocation_Quantity"): "Quantity",
        ("extent", "Allocation_Exchange"): "Allocation Exchange",
        ("extent", "Allocation_Shift"): "Allocation Shift",
        ("alternation_exchange", "Alternation_Exchange"): "Alternation Exchange",
        ("alternation_shift", "Alternation_Shift"): "Alternation Shift",
    }
    component_table = []
    for _, row in components[components["Class"] == name].iterrows():
        key = (row["Time_Interval"], row["Component"])
        if key in component_names:
            component_table.append(
                {
                    "component": component_names[key],
                    "gain_pixels": int(row["Gain"]),
                    "loss_pixels": int(row["Loss"]),
                },
            )

    # 4. Transitions between the first and last time points
    extent = cached["tables"]["aggregated_matrices"]["extent"]
    labels = {
        class_code: info["name"]
        for class_code, info in analyzer_class.class_labels_dict.items()
    }
    transition_table = []
    for other in extent.index:
        pairs = [(code, code)] if other == code else [(code, other), (other, code)]
        for from_code, to_code in pairs:
            pixels = int(extent.loc[from_code, to_code])
            if pixels > 0:
                transition_table.append(
                    {
                        "from_class": labels.get(from_code, str(from_code)),
                        "to_class": labels.get(to_code, str(to_code)),
                        "pixels": pixels,
                    },
                )

    def _section(
        rows: list[dict],
    ) -> dict:
        rows = [
            add_area_columns(row)
            for row in rows
        ]
        return {
            "columns": list(rows[0]) if rows else [],
            "rows": [
                list(row.values())
                for row in rows
            ],
        }

    return build_tool_result(
        analyzer,
        size_rows,
        class_name=name,
        change=_section(change_table),
        components=_section(component_table),
        transitions=_section(transition_table),
        note=(
            "rows hold the size at the first and last time points. The gain "
            "and loss in change sum every time interval. components and "
            "transitions refer to the whole time extent; the transition with "
            "from_class equal to to_class counts locations in the class at "
            "both the first and last time points, including locations that "
            "changed and came back, so it is not persistence through every "
            "time point."
        ),
    )

def plot_historical_class_sizes(specific_years: list[int] = None) -> dict:
    """
    Shows a stacked bar chart of the class sizes at each time point and
    returns the numbers plotted, so no data tool is needed afterwards.

    Parameters
    ----------
    specific_years : list of int, optional
        Years to plot. For a range such as 1990 to 2000, pass every year of
        the range. Leave empty to plot every year.
    """
    error = check_specific_years(analyzer_class, specific_years)
    if error:
        return error
    chart = analyzer_class.plot_class_sizes(specific_years=specific_years)
    return build_tool_result(
        analyzer_class,
        class_size_rows(analyzer_class, specific_years=specific_years),
        chart_description=chart,
    )

# --- Number of Changes Tools ---
def get_number_of_changes_per_interval() -> dict:
    """
    Returns, for each time interval, the number of locations that change
    class in that interval, split by the number of changes each location has
    over the whole time extent.
    """
    return build_tool_result(
        analyzer_changes,
        number_of_changes_interval_rows(analyzer_changes),
        note=NUMBER_OF_CHANGES_NOTE,
    )

def get_overall_number_of_changes() -> dict:
    """
    Returns the number of locations, and their percent of the study area,
    that change class once, twice, and so on over the whole time extent.
    Locations with zero changes are not listed.
    """
    return build_tool_result(
        analyzer_changes,
        overall_changes_rows(analyzer_changes),
    )

def plot_interval_changes_chart(start_year: int = None, end_year: int = None) -> dict:
    """
    Shows a stacked bar chart of the changing locations in each time
    interval, split by the number of changes of each location over the whole
    time extent, and returns the numbers plotted.

    Parameters
    ----------
    start_year, end_year : int, optional
        Show only the intervals between these years. Give both or neither.
        The number of changes still refers to the whole time extent; for the
        changes inside a custom time extent, use
        plot_custom_extent_overall_changes_chart.
    """
    error = check_year_range(analyzer_changes, start_year, end_year)
    if error:
        return error
    chart = analyzer_changes.plot_interval_changes(start_year=start_year, end_year=end_year)
    rows = number_of_changes_interval_rows(analyzer_changes, start_year, end_year)
    if not rows:
        return {"error": chart}
    return build_tool_result(
        analyzer_changes,
        rows,
        chart_description=chart,
        note=NUMBER_OF_CHANGES_NOTE,
    )

def plot_overall_changes_chart(metric: str = "percentage") -> dict:
    """
    Shows a bar chart of the locations by number of changes over the whole
    time extent and returns the numbers plotted.

    Parameters
    ----------
    metric : str
        'percentage' of the study area, the default, or 'pixels', the number
        of locations.
    """
    error = check_metric(metric)
    if error:
        return error
    chart = analyzer_changes.plot_overall_changes(metric=metric)
    return build_tool_result(
        analyzer_changes,
        overall_changes_rows(analyzer_changes),
        chart_description=chart,
    )

def plot_custom_extent_overall_changes_chart(start_year: int, end_year: int, metric: str = "percentage") -> dict:
    """
    Runs the analysis again for a custom time extent, shows the chart of the
    locations by number of changes inside it, and returns its numbers. The
    study area is recomputed for these years only.

    Parameters
    ----------
    start_year, end_year : int
        First and last years of the custom time extent.
    metric : str
        'percentage' of the study area, the default, or 'pixels', the number
        of locations.
    """
    error = check_metric(metric) or check_year_range(analyzer_changes, start_year, end_year)
    if error:
        return error
    custom_analyzer = QuestCustomExtentChanges(QUEST_CONFIG, start_year=start_year, end_year=end_year)
    chart = custom_analyzer.plot_overall_changes(metric=metric)
    return build_tool_result(
        custom_analyzer,
        overall_changes_rows(custom_analyzer),
        chart_description=chart,
    )

# --- Trajectory Tools ---
def get_trajectory_per_interval() -> dict:
    """
    Returns, for each time interval, the number of changing locations split
    by the trajectory of each location, trajectories 2 to 5. Locations of
    trajectory 1 never change, so they are not listed.
    """
    return build_tool_result(
        analyzer_traj,
        trajectory_interval_rows(analyzer_traj),
    )

def get_overall_trajectories() -> dict:
    """
    Returns the number of locations, and their percent of the study area, in
    each of the five trajectories over the whole time extent. The five add
    up to the study area.
    """
    return build_tool_result(
        analyzer_traj,
        overall_trajectory_rows(analyzer_traj),
    )

def plot_trajectory_interval_chart(start_year: int = None, end_year: int = None) -> dict:
    """
    Shows a stacked bar chart of the changing locations in each time
    interval by trajectory and returns the numbers plotted.

    Parameters
    ----------
    start_year, end_year : int, optional
        Show only the intervals between these years. Give both or neither.
    """
    error = check_year_range(analyzer_traj, start_year, end_year)
    if error:
        return error
    chart = analyzer_traj.plot_trajectory_intervals(start_year=start_year, end_year=end_year)
    rows = trajectory_interval_rows(analyzer_traj, start_year, end_year)
    if not rows:
        return {"error": chart}
    return build_tool_result(
        analyzer_traj,
        rows,
        chart_description=chart,
    )

def plot_trajectory_overall_chart(metric: str = "percentage") -> dict:
    """
    Shows a bar chart of trajectories 2 to 5 over the whole time extent and
    returns trajectories 1 to 5, with in_chart false for trajectory 1, which
    the chart leaves out.

    Parameters
    ----------
    metric : str
        'percentage' of the study area, the default, or 'pixels', the number
        of locations.
    """
    error = check_metric(metric)
    if error:
        return error
    chart = analyzer_traj.plot_overall_trajectories(metric=metric)
    return build_tool_result(
        analyzer_traj,
        overall_trajectory_rows(analyzer_traj, for_chart=True),
        chart_description=chart,
    )

def plot_custom_extent_overall_trajectory_chart(start_year: int, end_year: int, metric: str = "percentage") -> dict:
    """
    Runs the analysis again for a custom time extent, shows the chart of
    trajectories 2 to 5 inside it, and returns trajectories 1 to 5, with
    in_chart false for trajectory 1. The study area
    is recomputed for these years only.

    Parameters
    ----------
    start_year, end_year : int
        First and last years of the custom time extent.
    metric : str
        'percentage' of the study area, the default, or 'pixels', the number
        of locations.
    """
    error = check_metric(metric) or check_year_range(analyzer_traj, start_year, end_year)
    if error:
        return error
    custom_analyzer = QuestCustomExtentTrajectory(QUEST_CONFIG, start_year=start_year, end_year=end_year)
    chart = custom_analyzer.plot_overall_trajectories(metric=metric)
    return build_tool_result(
        custom_analyzer,
        overall_trajectory_rows(custom_analyzer, for_chart=True),
        chart_description=chart,
    )

# --- Transition Matrices Tools ---
def period_analyzer(
    analyzer,
    start_year: int | None,
    end_year: int | None,
):
    """
    Return the analyzer of the whole time extent, or a new analyzer of the
    same module restricted to the years from start_year to end_year.
    """
    if start_year is None and end_year is None:
        return analyzer

    custom = type(analyzer)(
        QUEST_CONFIG,
    )
    custom.restrict_years(
        start_year=start_year,
        end_year=end_year,
    )
    return custom


def get_transition_matrix(matrix_key: str, start_year: int = None, end_year: int = None) -> dict:
    """
    Returns a transition matrix as rows of from_class, to_class and the
    number of locations. Rows where from_class equals to_class hold
    persistence.

    Parameters
    ----------
    matrix_key : str
        One of 'time_intervals', the sum of the transitions of every
        interval; 'extent', from the first to the last time point;
        'allocation_exchange' and 'quantity_allocation_shift', the two parts
        of the extent matrix; 'alternation_exchange'; 'alternation_shift';
        'indirect_extent'; 'trajectory_1' to 'trajectory_5', the sum of the
        transitions of the locations of one trajectory; or
        'interval_<start>-<end>' for one interval. Call get_dataset_metadata
        for the exact list; never guess a key.
    start_year, end_year : int, optional
        Run the analysis again for the years between them, a custom time
        extent with its own study area. Give both or neither; without them,
        the result covers the whole time extent.
    """
    error = check_year_range(analyzer_trans, start_year, end_year)
    if error:
        return error
    analyzer = period_analyzer(analyzer_trans, start_year, end_year)
    matrices = analyzer.compute_transition_matrices()
    if matrix_key in matrices:
        return build_tool_result(
            analyzer,
            matrix_rows(analyzer, matrices[matrix_key]),
            matrix_key=matrix_key,
        )
    return {
        "error": (
            f"Matrix '{matrix_key}' not found for {analyzer.years[0]} to "
            f"{analyzer.years[-1]}. Call get_dataset_metadata for the valid "
            f"keys; interval keys must fall inside the time extent."
        ),
    }

def plot_transition_heatmap(matrix_key: str, start_year: int = None, end_year: int = None) -> dict:
    """
    Shows a heatmap of a transition matrix and returns the numbers plotted.

    Parameters
    ----------
    matrix_key : str
        One of 'time_intervals', the sum of the transitions of every
        interval; 'extent', from the first to the last time point;
        'allocation_exchange' and 'quantity_allocation_shift', the two parts
        of the extent matrix; 'alternation_exchange'; 'alternation_shift';
        'indirect_extent'; 'trajectory_1' to 'trajectory_5', the sum of the
        transitions of the locations of one trajectory; or
        'interval_<start>-<end>' for one interval. Call get_dataset_metadata
        for the exact list; never guess a key.
    start_year, end_year : int, optional
        Run the analysis again for the years between them, a custom time
        extent with its own study area. Give both or neither; without them,
        the result covers the whole time extent.
    """
    error = check_year_range(analyzer_trans, start_year, end_year)
    if error:
        return error
    analyzer = period_analyzer(analyzer_trans, start_year, end_year)
    chart = analyzer.plot_matrix_heatmap(matrix_key=matrix_key)
    matrices = analyzer.compute_transition_matrices()
    if matrix_key not in matrices:
        return {"error": chart}
    return build_tool_result(
        analyzer,
        matrix_rows(analyzer, matrices[matrix_key]),
        chart_description=chart,
        matrix_key=matrix_key,
    )

# --- Change Components Tools ---
def get_change_components_table(start_year: int = None, end_year: int = None) -> dict:
    """
    Returns the full change components table: the gain and loss of each
    class for each component, for every time interval and for the extent,
    alternation exchange and alternation shift. Prefer the component chart
    tools, which return smaller summaries, unless the user asks for the
    whole table.

    Parameters
    ----------
    start_year, end_year : int, optional
        Run the analysis again for the years between them, a custom time
        extent with its own study area. Give both or neither; without them,
        the result covers the whole time extent.
    """
    error = check_year_range(analyzer_comp, start_year, end_year)
    if error:
        return error
    analyzer = period_analyzer(analyzer_comp, start_year, end_year)
    return build_tool_result(
        analyzer,
        component_rows(analyzer, analyzer.compute_change_components_table()),
    )

def plot_intervals_components(start_year: int = None, end_year: int = None) -> dict:
    """
    Shows a stacked bar chart of Quantity, Exchange and Shift in each time
    interval, summed over the classes, and returns the numbers plotted.

    Parameters
    ----------
    start_year, end_year : int, optional
        Run the analysis again for the years between them, a custom time
        extent with its own study area. Give both or neither; without them,
        the result covers the whole time extent.
    """
    error = check_year_range(analyzer_comp, start_year, end_year)
    if error:
        return error
    analyzer = period_analyzer(analyzer_comp, start_year, end_year)
    chart = analyzer.plot_change_components_intervals()
    return build_tool_result(
        analyzer,
        component_interval_rows(analyzer),
        chart_description=chart,
    )

def plot_overall_components(start_year: int = None, end_year: int = None) -> dict:
    """
    Shows a single stacked bar of the change over the time extent: Quantity,
    Allocation Exchange, Allocation Shift, Alternation Exchange and
    Alternation Shift. Returns the size of each component.

    Parameters
    ----------
    start_year, end_year : int, optional
        Run the analysis again for the years between them, a custom time
        extent with its own study area. Give both or neither; without them,
        the result covers the whole time extent.
    """
    error = check_year_range(analyzer_comp, start_year, end_year)
    if error:
        return error
    analyzer = period_analyzer(analyzer_comp, start_year, end_year)
    chart = analyzer.plot_change_components_overall()
    return build_tool_result(
        analyzer,
        component_overall_rows(analyzer),
        chart_description=chart,
    )

def plot_class_components(start_year: int = None, end_year: int = None) -> dict:
    """
    Shows the gains and losses of each class by component over the time
    extent and returns the numbers plotted.

    Parameters
    ----------
    start_year, end_year : int, optional
        Run the analysis again for the years between them, a custom time
        extent with its own study area. Give both or neither; without them,
        the result covers the whole time extent.
    """
    error = check_year_range(analyzer_comp, start_year, end_year)
    if error:
        return error
    analyzer = period_analyzer(analyzer_comp, start_year, end_year)
    chart = analyzer.plot_change_components_by_class()
    return build_tool_result(
        analyzer,
        component_class_rows(analyzer),
        chart_description=chart,
    )

# --- Maps Tools ---
def plot_number_of_changes_spatial_map() -> dict:
    """
    Shows the map of the number of changes of each location over the whole
    time extent and returns the locations by number of changes.
    """
    chart = analyzer_maps.plot_number_of_changes_map()
    return build_tool_result(
        analyzer_maps,
        overall_changes_rows(analyzer_changes),
        chart_description=chart,
    )

def plot_trajectory_spatial_map() -> dict:
    """
    Shows the map of the trajectory of each location and returns the
    locations of each trajectory.
    """
    chart = analyzer_maps.plot_trajectory_map()
    return build_tool_result(
        analyzer_maps,
        overall_trajectory_rows(analyzer_maps),
        chart_description=chart,
    )

def plot_change_component_map(component: str) -> dict:
    """
    Shows the map of one change component over the whole time extent and
    returns the number of locations of each map value, plus two totals: the
    locations with a value above 0, and the changes in the component, the
    sum of value times locations. Use these totals instead of adding rows.

    Parameters
    ----------
    component : str
        'extent', where 1 marks a location whose class differs between the
        first and last time points and 0 a location with the same class;
        'alternation_exchange' or 'alternation_shift', where each value is
        the number of changes of the location in that component.
    """
    if component not in analyzer_maps.COMPONENT_MAPS:
        return {
            "error": (
                f"The component '{component}' has no map. Use 'extent', "
                f"'alternation_exchange' or 'alternation_shift'."
            ),
        }

    chart = analyzer_maps.plot_component_map(component)
    counts = analyzer_maps.count_raster_values(
        analyzer_maps.compute_component_raster(component),
    )
    study_area = analyzer_maps.get_quest_results()["study_area_pixels"]
    rows = [
        {
            "value": value,
            "pixels": count,
            "percent": _percent(count, study_area),
        }
        for value, count in sorted(counts.items())
    ]
    # Totals, so the agent never adds the rows itself
    totals = [
        {
            "measure": "locations with a value above 0",
            "pixels": sum(
                count
                for value, count in counts.items()
                if value > 0
            ),
        },
        {
            "measure": "changes in this component, the sum of value times locations",
            "pixels": sum(
                value * count
                for value, count in counts.items()
            ),
        },
    ]
    for row in totals:
        row["percent"] = _percent(row["pixels"], study_area)
    totals = [
        add_area_columns(row)
        for row in totals
    ]

    if component == "extent":
        note = "value 1 is extent change and value 0 is no extent change."
    else:
        note = (
            "value is the number of changes of the location in this "
            "component; value 0 means none."
        )

    return build_tool_result(
        analyzer_maps,
        rows,
        chart_description=chart,
        component=component,
        totals={
            "columns": list(totals[0]),
            "rows": [
                list(row.values())
                for row in totals
            ],
        },
        note=note,
    )


# 5. Unified Tool List, each tool wrapped so its calls are recorded
all_tools = [
    get_dataset_metadata,
    get_class_pixels_over_time,
    get_class_summary,
    plot_historical_class_sizes,
    get_number_of_changes_per_interval,
    get_overall_number_of_changes,
    plot_interval_changes_chart,
    plot_overall_changes_chart,
    plot_custom_extent_overall_changes_chart,
    get_trajectory_per_interval,
    get_overall_trajectories,
    plot_trajectory_interval_chart,
    plot_trajectory_overall_chart,
    plot_custom_extent_overall_trajectory_chart,
    get_transition_matrix,
    plot_transition_heatmap,
    get_change_components_table,
    plot_intervals_components,
    plot_overall_components,
    plot_class_components,
    plot_number_of_changes_spatial_map,
    plot_trajectory_spatial_map,
    plot_change_component_map,
]
all_tools = [
    log_tool_call(tool)
    for tool in all_tools
]

# 6. Agent Configuration
# Instruction given to the agent, with the definitions of the QUEST method
SYSTEM_INSTRUCTION = """
You are QuestEngine, an analyst of land cover change that uses the QUEST
method on the maps loaded in this notebook. Answer only questions about
these data. For any other question, refuse and say that your scope is the
QUEST analysis of these data.

How to work
- Every number in an answer must come from a tool result in this
  conversation. You may add, subtract or compare numbers that tools
  returned, but never estimate or compute new statistics.
- Use get_dataset_metadata for the years, intervals, classes and matrix keys
  of the dataset. Never guess them.
- For a question about one class, use get_class_summary, which returns
  its sizes, gains, losses, components and transitions in one result.
- Chart and map tools already return the numbers they show. After a chart or
  map tool, do not call a data tool for the same numbers.
- Results count locations, one location per pixel of the rasters, and give
  their percent of the study area. Columns and options named "pixels" hold
  these counts of locations. In answers, write "locations", not "pixels".
  Every result also gives areas in square kilometers and hectares, in
  columns ending in _km2 and _ha; use them for questions about area, and
  when a result has an area_note, mention that areas are approximate.
  The study area is the set of locations that are valid and belong to a
  listed class at every time point, so a custom time extent can have a
  different study area.
- When a tool returns an error, explain it to the user in plain words with
  the valid options it gives.
- When a tool returns text enclosed in dashed lines, output that text
  exactly, keeping its line breaks.
- Answer in the language of the question. Keep class names and matrix keys
  exactly as they appear in the data, since tools need them unchanged.

QUEST definitions
- Time interval: the step between two consecutive time points. A location
  changes in an interval when its class differs between the two time points.
- Number of changes: the number of intervals in which a location changes.
- Extent: the transition from the class at the first time point to the class
  at the last time point. A location has extent change when the two classes
  differ.
- Alternation: the changes of a location beyond its extent change.
  Alternation Exchange counts the changes that form pairs of opposite
  transitions in the same location, such as a change from one class to
  another and a later change back. Alternation Shift counts the rest of the
  alternation.
- Indirect extent: an extent transition that never happens in a single
  interval, because the location passes through other classes. The change
  components exclude it from Alternation Shift.
- Components of the extent: Quantity is the net gain or net loss of a class
  between the first and last time points. Allocation Exchange counts the
  locations whose extent transitions form pairs of opposite transitions
  between two classes. Allocation Shift is the rest of the extent change.
- Components of each time interval: Quantity, Exchange and Shift, with the
  same meaning, computed from the transitions of that interval.
- Trajectories: 1, no change; 2, more than one change with the same class at
  the first and last time points; 3, exactly one change; 4, more than two
  changes, with the transition from the first to the last class happening in
  a single interval; 5, more than one change, without that direct
  transition.
"""

# Retry a failed model call up to 3 times on server errors, waiting 5 s,
# then up to 30 s. Rate limit errors, 429, are left to wait_for_rate_limit.
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            attempts=3,
            initial_delay=5.0,
            max_delay=30.0,
            http_status_codes=[
                500,
                503,
                504,
            ],
        ),
    ),
)

chat = client.chats.create(
    model=GEMINI_MODEL,
    config=types.GenerateContentConfig(
        tools=all_tools,
        temperature=0.1,
        system_instruction=SYSTEM_INSTRUCTION,
    )
)

# 7. Ask a question, time the answer and write one line per question to the
#    chat log
CHAT_LOG_PATH = os.path.join(
    OUTPUT_PATH,
    "logs",
    f"chat_{datetime.now():%Y%m%d_%H%M%S}.jsonl",
)
os.makedirs(
    os.path.dirname(CHAT_LOG_PATH),
    exist_ok=True,
)


_model_call_times = []


def wait_for_rate_limit(
    expected_calls: int = 3,
) -> float:
    """
    Wait before a question when the model calls of the last minute, plus the
    calls expected for the new question, would pass REQUESTS_PER_MINUTE.

    Parameters
    ----------
    expected_calls : int, optional
        Model calls expected for one question. Questions in the tests made
        1 to 4 calls. Default is 3.

    Returns
    -------
    float
        Seconds waited.
    """
    if REQUESTS_PER_MINUTE <= 0:
        return 0.0

    now = time.time()
    _model_call_times[:] = [
        call_time
        for call_time in _model_call_times
        if now - call_time < 60
    ]
    excess = (
        len(_model_call_times)
        + min(expected_calls, REQUESTS_PER_MINUTE)
        - REQUESTS_PER_MINUTE
    )
    if excess <= 0:
        return 0.0

    # Wait until enough of the oldest calls are more than a minute old
    wait = 60 - (now - _model_call_times[excess - 1]) + 1
    print(
        f"Waiting {wait:.0f} s to stay under {REQUESTS_PER_MINUTE} model "
        f"requests per minute...",
    )
    time.sleep(
        wait,
    )
    return wait


def ask_quest(
    question: str,
) -> dict:
    """
    Send a question to the agent and record how it was answered.

    Parameters
    ----------
    question : str
        Question typed by the user.

    Returns
    -------
    dict
        Question, answer or error, total time, tokens, number of model calls
        and the tool calls, as written to CHAT_LOG_PATH.

    Notes
    -----
    The token counts come from the last model call of the answer. When the
    agent calls tools, the earlier model calls are not included.
    """
    waited = wait_for_rate_limit()
    _tool_calls.clear()
    record = {
        "timestamp": datetime.now().isoformat(
            timespec="seconds",
        ),
        "model": GEMINI_MODEL,
        "question": question,
        "rate_limit_wait_seconds": round(waited, 1),
    }
    start = time.perf_counter()

    try:
        response = chat.send_message(
            question,
        )
        usage = response.usage_metadata
        history = response.automatic_function_calling_history or []

        # The history holds the whole chat, so count only the model calls
        # made after the current question
        question_index = max(
            (
                index
                for index, content in enumerate(history)
                if content.role == "user"
                and any(part.text for part in content.parts or [])
            ),
            default=-1,
        )
        record.update(
            {
                "status": "ok",
                "answer": response.text,
                "model_calls": 1 + sum(
                    1
                    for content in history[question_index + 1:]
                    if content.role == "model"
                ),
                "input_tokens": getattr(usage, "prompt_token_count", None),
                "output_tokens": getattr(usage, "candidates_token_count", None),
                "thinking_tokens": getattr(usage, "thoughts_token_count", None),
            },
        )
    except Exception as error:
        record.update(
            {
                "status": "error",
                "error_code": getattr(error, "code", None),
                "error": f"{type(error).__name__}: {error}",
                "user_message": describe_api_error(error),
            },
        )

    record["seconds"] = round(
        time.perf_counter() - start,
        3,
    )
    _model_call_times.extend(
        [time.time()] * record.get("model_calls", 1),
    )
    record["tool_calls"] = list(
        _tool_calls,
    )

    with open(
        CHAT_LOG_PATH,
        "a",
        encoding="utf-8",
    ) as log_file:
        log_file.write(
            json.dumps(
                record,
                ensure_ascii=False,
                default=str,
            ) + "\n",
        )

    return record


def describe_api_error(
    error: Exception,
) -> str:
    """
    Write a short message, for the user, about an error of the Gemini API.

    Parameters
    ----------
    error : Exception
        Error raised while sending a question to the model.

    Returns
    -------
    str
        What happened and what the user can do.
    """
    code = getattr(error, "code", None)
    message = str(getattr(error, "message", None) or error)

    if code == 402:
        return (
            "The API credits of this key are used up. Add credits in AI "
            "Studio, or use a key from a project on the free tier."
        )
    if code == 404:
        suggestion = re.search(
            r"use models/([\w.\-]+)",
            message,
        )
        text = (
            f"The model {GEMINI_MODEL} is not available for this key. Change "
            f"GEMINI_MODEL in cell 2.1"
        )
        if suggestion:
            text += f", for example to {suggestion.group(1)}"
        return text + ", and run cells 2.1 and 5.1 again."
    if code == 429:
        return (
            "The request limit of this key was reached. Wait about one "
            "minute and ask again. If it happens again, the daily limit may "
            "be used up."
        )
    if code in (500, 502, 503, 504):
        return (
            f"The model {GEMINI_MODEL} is overloaded or unavailable right "
            f"now. Try again in a few minutes, or change GEMINI_MODEL in cell "
            f"2.1."
        )
    if code in (400, 401, 403) and "key" in message.lower():
        return (
            "The API key was rejected. Check the GEMINI_API_KEY secret, "
            "restart the session and run all cells again."
        )
    if code is None:
        return (
            f"The question could not be sent to the Gemini API, because of "
            f"{type(error).__name__}: {error}. Check the internet connection "
            f"and ask again."
        )

    return f"The Gemini API returned error {code}: {message}"


def format_tool_calls(
    tool_calls: list[dict],
) -> str:
    """
    Write the tool calls of one answer as a short line for the chat.

    Parameters
    ----------
    tool_calls : list of dict
        Tool calls recorded by `log_tool_call`.

    Returns
    -------
    str
        One line listing each tool, its arguments, run time and errors.
    """
    if not tool_calls:
        return "Tools: none"

    calls = []
    for call in tool_calls:
        arguments = call["arguments"]
        if isinstance(arguments, dict):
            arguments = ", ".join(
                f"{key}={value!r}"
                for key, value in arguments.items()
            )
        else:
            arguments = ", ".join(
                repr(value)
                for value in arguments
            )
        text = f"{call['tool']}({arguments}) {call['seconds']:.1f} s"
        if call["status"] == "error":
            text += f" [error: {call['error']}]"
        calls.append(
            text,
        )

    return "Tools: " + " · ".join(calls)


# 8. Continuous Chat Interface
print(
    "Hello! I am QuestEngine, your friendly agentic AI for spatial analysis.\n"
    "I'm here to help you explore your land cover data using the QUEST methodology.\n\n"
    "If you are not sure where to start, I suggest this step-by-step workflow:\n"
    " 1. Explore the time series of your input raster maps (ask about class sizes/extents over time).\n"
    " 2. Look into the Number of Changes and Trajectories to see location-level dynamics.\n"
    " 3. Dive into the Transition Matrices and Change Components for detailed mechanics of change.\n\n"
    "Feel free to ask for data, graphics, or maps whenever you're ready!\n"
    "(Type 'quit' to exit)\n" +
    "-" * 60
)

print(
    f"The questions, answers and tool calls are saved to {CHAT_LOG_PATH}\n" +
    "-" * 60
)

while True:
    question = input("You: ")

    if question.strip().lower() == "quit":
        break

    if not question.strip():
        continue

    record = ask_quest(
        question,
    )

    if record["status"] == "ok":
        print(f"QuestEngine: {record['answer']}")
    else:
        print(f"QuestEngine Error: {record['user_message']}")

    if SHOW_TOOL_CALLS:
        print(
            format_tool_calls(
                record["tool_calls"],
            ),
        )

    print("-" * 60)
    time.sleep(4)